In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ============================================================
# 04_swin_loss_screening_development_only.ipynb
# COMPLETE SINGLE-CELL, DEVELOPMENT-ONLY, RESUME-SAFE,
# GOOGLE-DRIVE-I/O-SAFE VERSION
# ============================================================
#
# Purpose
# -------
# 1. Screen five loss formulations using pretrained Swin-Base.
# 2. Reuse the frozen 478-patient, five-fold assignment.
# 3. Use modal_low as the primary derived patient reference.
# 4. Use mean_probability_argmax as the locked aggregation.
# 5. Select checkpoints by validation patient-level metrics.
# 6. Rank losses using pooled development OOF metrics only.
# 7. Shortlist two losses for multi-seed confirmation.
#
# Test-set policy
# ---------------
# The official LIMUC test set is never defined, loaded,
# evaluated, or used for model selection.
#
# Resume policy
# -------------
# - Completed experiments are reused.
# - Completed folds are reused.
# - Interrupted folds resume after the last completed epoch.
# - Only the interrupted epoch is repeated.
#
# Drive-I/O protection
# --------------------
# - Runtime DataLoader workers are set to zero.
# - Image reads are retried after transient Drive errors.
# - Successfully read images are cached locally in /content.
# - Canonical Drive paths remain in all prediction CSV files.
#
# Training workload
# -----------------
# 5 losses × 5 folds × 1 seed = 25 fold trainings.
# ============================================================


# ============================================================
# 0. Install dependencies
# ============================================================

%pip -q install timm==1.0.28 scikit-learn==1.6.1 pandas==2.2.2 Pillow==11.3.0


# ============================================================
# 1. Imports
# ============================================================

import gc
import io
import os
import json
import math
import copy
import time
import random
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

from IPython.display import display


ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")


# ============================================================
# 2. Development-only safety check
# ============================================================

FORBIDDEN_TEST_OBJECTS = [
    "TEST_PATH",
    "TEST_ROOT",
    "test_df",
    "test_dataset",
    "test_loader",
    "test_predictions",
    "test_patient_predictions",
    "ensemble_test_df",
    "ensemble_test_patient_df",
    "test_fold_prob_dfs",
    "all_test_fold_prob_df",
    "ensemble_test_metrics",
    "ensemble_test_patient_metrics",
]

present_test_objects = [
    name
    for name in FORBIDDEN_TEST_OBJECTS
    if name in globals()
]

if present_test_objects:
    raise RuntimeError(
        "Official-test objects exist in the current runtime: "
        f"{present_test_objects}\n"
        "Restart the runtime and run only this cell."
    )

print("Development-only safety check passed.")
print("Official LIMUC test data are not defined or loaded.")


# ============================================================
# 3. Mount Google Drive
# ============================================================

from google.colab import drive

(drive.mount(
    "/content/drive",
    force_remount=True,
) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)


# ============================================================
# 4. Paths
# ============================================================

DEVELOPMENT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/train_and_validation_sets')
)

SCREENING_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/01_vit_loss_screening_development_only')
)

CONFIRMATION_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/02_vit_loss_multiseed_confirmation_epoch_resume_v2')
)

AGGREGATION_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/03_patient_aggregation_and_reference_sensitivity')
)

# Keep this path unchanged so completed experiments, folds,
# and the interrupted fold checkpoint are reused.
OUTPUT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/04_swin_loss_screening_development_only')
)

MANIFEST_PATH = (
    SCREENING_ROOT
    / "development_image_manifest_with_folds.csv"
)

FOLD_ASSIGNMENT_PATH = (
    SCREENING_ROOT
    / "patient_fold_assignment.csv"
)

REFERENCE_AUDIT_PATH = (
    SCREENING_ROOT
    / "patient_reference_rule_sensitivity.csv"
)

SCREENING_PLAN_PATH = (
    SCREENING_ROOT
    / "analysis_plan_frozen_before_training.json"
)

CONFIRMATION_PLAN_PATH = (
    CONFIRMATION_ROOT
    / "analysis_plan_frozen_before_confirmation.json"
)

AGGREGATION_PLAN_PATH = (
    AGGREGATION_ROOT
    / "analysis_plan_frozen_before_aggregation.json"
)

LOCKED_AGGREGATION_PATH = (
    AGGREGATION_ROOT
    / "locked_patient_aggregation_rule.json"
)

required_paths = [
    DEVELOPMENT_ROOT,
    SCREENING_ROOT,
    CONFIRMATION_ROOT,
    AGGREGATION_ROOT,
    MANIFEST_PATH,
    FOLD_ASSIGNMENT_PATH,
    REFERENCE_AUDIT_PATH,
    SCREENING_PLAN_PATH,
    CONFIRMATION_PLAN_PATH,
    AGGREGATION_PLAN_PATH,
    LOCKED_AGGREGATION_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist:\n{required_path}"
        )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nDevelopment root:", DEVELOPMENT_ROOT)
print("Frozen development files:", SCREENING_ROOT)
print("Locked aggregation files:", AGGREGATION_ROOT)
print("Swin screening output:", OUTPUT_ROOT)


# ============================================================
# 5. Frozen scientific protocol configuration
# ============================================================

PROTOCOL_VERSION = (
    "swin_loss_screening_development_only_v1"
)

PRIMARY_REFERENCE_RULE = "modal_low"

LOCKED_AGGREGATION_RULE = (
    "mean_probability_argmax"
)

SCREENING_SEED = 42

NUM_CLASSES = 4
NUM_FOLDS = 5

# Scientific training configuration retained from the initial
# frozen Swin screening analysis plan.
TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2

EFFECTIVE_TRAIN_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

VALIDATION_BATCH_SIZE = 8

# Retain this value in the frozen scientific plan.
NUM_WORKERS = 2

IMAGE_SIZE = 384

EPOCHS = 25
LEARNING_RATE = 1e-4
MIN_LR_RATIO = 0.05
WARMUP_EPOCHS = 3
WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.02
DROP_RATE = 0.10
DROP_PATH_RATE = 0.05

EARLY_STOP_PATIENCE = 8
GRAD_CLIP_NORM = 1.0

USE_MIXED_PRECISION = True
USE_CLASS_WEIGHTS_IN_CE = True
USE_WEIGHTED_SAMPLER = False

USE_TTA = True
TTA_HORIZONTAL_FLIP = True

BOOTSTRAP_REPETITIONS = 5000
BOOTSTRAP_SEED = 20260804

RESUME_COMPLETED_FOLDS = True
RESUME_INTERRUPTED_FOLDS = True
SKIP_COMPLETED_EXPERIMENTS = True

EPOCH_CHECKPOINT_INTERVAL = 1


# ============================================================
# 6. Runtime-only Google Drive I/O safeguards
# ============================================================

# These settings do not alter the scientific model protocol.
# They address the transient Google Drive Errno 5 read failure.

RUNTIME_NUM_WORKERS = 0

USE_LOCAL_IMAGE_CACHE = True

LOCAL_IMAGE_CACHE_ROOT = Path(
    "/content/LIMUC_development_image_cache"
)

LOCAL_IMAGE_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

IMAGE_IO_MAX_RETRIES = 12
IMAGE_IO_INITIAL_DELAY_SECONDS = 1.0
IMAGE_IO_MAX_DELAY_SECONDS = 8.0

DRIVE_WRITE_MAX_RETRIES = 5
DRIVE_WRITE_INITIAL_DELAY_SECONDS = 1.0

print(
    "\nConfigured workers in frozen analysis plan:",
    NUM_WORKERS,
)

print(
    "Runtime DataLoader workers:",
    RUNTIME_NUM_WORKERS,
)

print(
    "Local image cache enabled:",
    USE_LOCAL_IMAGE_CACHE,
)

print(
    "Local image cache root:",
    LOCAL_IMAGE_CACHE_ROOT,
)


# ============================================================
# 7. Resolve pretrained Swin-Base 384 identifier
# ============================================================

PREFERRED_SWIN_MODEL_NAMES = [
    "swin_base_patch4_window12_384.ms_in22k_ft_in1k",
    "swin_base_patch4_window12_384",
]


def resolve_pretrained_swin_model_name():
    pretrained_models = set(
        timm.list_models(
            pretrained=True
        )
    )

    for model_name in PREFERRED_SWIN_MODEL_NAMES:
        if model_name in pretrained_models:
            return model_name

    compatible_models = sorted([
        model_name
        for model_name in pretrained_models
        if model_name.startswith(
            "swin_base_patch4_window12_384"
        )
    ])

    if compatible_models:
        return compatible_models[0]

    similar_models = sorted([
        model_name
        for model_name in pretrained_models
        if (
            model_name.startswith("swin_base")
            and "384" in model_name
        )
    ])

    raise RuntimeError(
        "No pretrained Swin-Base 384 model was found in "
        f"timm {timm.__version__}.\n"
        f"Similar models: {similar_models[:20]}"
    )


RESOLVED_SWIN_MODEL_NAME = (
    resolve_pretrained_swin_model_name()
)

print(
    "\nResolved pretrained Swin model:",
    RESOLVED_SWIN_MODEL_NAME,
)


# ============================================================
# 8. Loss-screening candidates
# ============================================================

EXPERIMENT_PRESETS = {
    "swin_L1_ce_only": {
        "display_name": "CE only",
        "loss_type": "ce",
        "ordinal_loss_weight": 0.0,
        "ordinal_alpha": 1.0,
    },

    "swin_L2_expected_value_ordinal_mae_w010": {
        "display_name": (
            "CE + expected-value ordinal MAE, lambda=0.10"
        ),
        "loss_type": "ce_expected_value_mae",
        "ordinal_loss_weight": 0.10,
        "ordinal_alpha": 1.0,
    },

    "swin_L3_expected_ordinal_distance_a1_w010": {
        "display_name": (
            "CE + expected ordinal-distance loss, "
            "alpha=1, lambda=0.10"
        ),
        "loss_type": "ce_expected_distance",
        "ordinal_loss_weight": 0.10,
        "ordinal_alpha": 1.0,
    },

    "swin_L4_expected_ordinal_distance_a2_w010": {
        "display_name": (
            "CE + expected ordinal-distance loss, "
            "alpha=2, lambda=0.10"
        ),
        "loss_type": "ce_expected_distance",
        "ordinal_loss_weight": 0.10,
        "ordinal_alpha": 2.0,
    },

    "swin_L5_expected_ordinal_distance_a1_w020": {
        "display_name": (
            "CE + expected ordinal-distance loss, "
            "alpha=1, lambda=0.20"
        ),
        "loss_type": "ce_expected_distance",
        "ordinal_loss_weight": 0.20,
        "ordinal_alpha": 1.0,
    },
}

EXPERIMENTS_TO_RUN = list(
    EXPERIMENT_PRESETS.keys()
)

BASE_TRAINING_CONFIG = {
    "model_name": RESOLVED_SWIN_MODEL_NAME,
    "pretrained": True,

    "num_classes": NUM_CLASSES,
    "num_folds": NUM_FOLDS,

    "image_size": IMAGE_SIZE,

    "train_batch_size": TRAIN_BATCH_SIZE,

    "gradient_accumulation_steps": (
        GRADIENT_ACCUMULATION_STEPS
    ),

    "effective_train_batch_size": (
        EFFECTIVE_TRAIN_BATCH_SIZE
    ),

    "validation_batch_size": (
        VALIDATION_BATCH_SIZE
    ),

    # Keep the frozen-plan value at two.
    "num_workers": NUM_WORKERS,

    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "min_lr_ratio": MIN_LR_RATIO,
    "warmup_epochs": WARMUP_EPOCHS,
    "weight_decay": WEIGHT_DECAY,

    "label_smoothing": LABEL_SMOOTHING,
    "drop_rate": DROP_RATE,
    "drop_path_rate": DROP_PATH_RATE,

    "early_stop_patience": (
        EARLY_STOP_PATIENCE
    ),

    "grad_clip_norm": GRAD_CLIP_NORM,

    "use_mixed_precision": (
        USE_MIXED_PRECISION
    ),

    "use_class_weights_in_ce": (
        USE_CLASS_WEIGHTS_IN_CE
    ),

    "use_weighted_sampler": (
        USE_WEIGHTED_SAMPLER
    ),

    "use_tta": USE_TTA,

    "tta_horizontal_flip": (
        TTA_HORIZONTAL_FLIP
    ),

    "primary_patient_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),

    "screening_seed": SCREENING_SEED,

    "bootstrap_repetitions": (
        BOOTSTRAP_REPETITIONS
    ),

    "bootstrap_seed": BOOTSTRAP_SEED,

    "epoch_checkpoint_interval": (
        EPOCH_CHECKPOINT_INTERVAL
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

print("\nSwin loss candidates:")

for experiment_name in EXPERIMENTS_TO_RUN:
    print(
        " -",
        experiment_name,
        ":",
        EXPERIMENT_PRESETS[
            experiment_name
        ]["display_name"],
    )

print("Screening seed:", SCREENING_SEED)

print(
    "Effective training batch size:",
    EFFECTIVE_TRAIN_BATCH_SIZE,
)


# ============================================================
# 9. File-writing and loading helpers
# ============================================================

def temporary_path(
    path: Path,
) -> Path:
    return path.with_name(
        path.name + ".temporary"
    )


def retry_delay(
    attempt_index: int,
    initial_delay: float,
    maximum_delay: float = 8.0,
) -> float:
    return min(
        maximum_delay,
        initial_delay
        * (attempt_index + 1),
    )


def atomic_json_save(
    payload: dict,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_WRITE_MAX_RETRIES
    ):
        temp_path = temporary_path(path)

        try:
            if temp_path.exists():
                temp_path.unlink()

            with open(
                temp_path,
                "w",
                encoding="utf-8",
            ) as file:
                json.dump(
                    payload,
                    file,
                    indent=2,
                    default=str,
                )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            delay = retry_delay(
                attempt_index,
                DRIVE_WRITE_INITIAL_DELAY_SECONDS,
            )

            print(
                f"JSON write retry "
                f"{attempt_index + 1}/"
                f"{DRIVE_WRITE_MAX_RETRIES}: "
                f"{path}"
            )

            time.sleep(delay)

    raise OSError(
        f"Could not save JSON file:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def atomic_csv_save(
    dataframe: pd.DataFrame,
    path: Path,
    index: bool = False,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_WRITE_MAX_RETRIES
    ):
        temp_path = temporary_path(path)

        try:
            if temp_path.exists():
                temp_path.unlink()

            dataframe.to_csv(
                temp_path,
                index=index,
            )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            delay = retry_delay(
                attempt_index,
                DRIVE_WRITE_INITIAL_DELAY_SECONDS,
            )

            print(
                f"CSV write retry "
                f"{attempt_index + 1}/"
                f"{DRIVE_WRITE_MAX_RETRIES}: "
                f"{path}"
            )

            time.sleep(delay)

    raise OSError(
        f"Could not save CSV file:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def atomic_torch_save(
    payload,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_WRITE_MAX_RETRIES
    ):
        temp_path = temporary_path(path)

        try:
            if temp_path.exists():
                temp_path.unlink()

            torch.save(
                payload,
                temp_path,
            )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            delay = retry_delay(
                attempt_index,
                DRIVE_WRITE_INITIAL_DELAY_SECONDS,
            )

            print(
                f"Checkpoint write retry "
                f"{attempt_index + 1}/"
                f"{DRIVE_WRITE_MAX_RETRIES}: "
                f"{path}"
            )

            time.sleep(delay)

    raise OSError(
        f"Could not save checkpoint:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def safe_torch_load(
    path: Path,
    map_location="cpu",
):
    last_error = None

    for attempt_index in range(
        DRIVE_WRITE_MAX_RETRIES
    ):
        try:
            try:
                return torch.load(
                    path,
                    map_location=map_location,
                    weights_only=False,
                )

            except TypeError:
                return torch.load(
                    path,
                    map_location=map_location,
                )

        except OSError as error:
            last_error = error

            delay = retry_delay(
                attempt_index,
                DRIVE_WRITE_INITIAL_DELAY_SECONDS,
            )

            print(
                f"Checkpoint read retry "
                f"{attempt_index + 1}/"
                f"{DRIVE_WRITE_MAX_RETRIES}: "
                f"{path}"
            )

            time.sleep(delay)

    raise OSError(
        f"Could not load checkpoint:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


for abandoned_file in OUTPUT_ROOT.rglob(
    "*.temporary"
):
    try:
        abandoned_file.unlink()
    except Exception:
        pass

for abandoned_file in LOCAL_IMAGE_CACHE_ROOT.rglob(
    "*.temporary"
):
    try:
        abandoned_file.unlink()
    except Exception:
        pass


# ============================================================
# 10. Reproducibility and RNG helpers
# ============================================================

def seed_everything(
    seed: int,
) -> None:
    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def seed_worker(
    worker_id: int,
) -> None:
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def as_cpu_byte_tensor(
    state,
    state_name: str,
):
    if state is None:
        raise ValueError(
            f"{state_name} is missing."
        )

    if isinstance(
        state,
        torch.Tensor,
    ):
        return (
            state.detach()
            .to(
                device="cpu",
                dtype=torch.uint8,
            )
            .contiguous()
        )

    if isinstance(
        state,
        np.ndarray,
    ):
        return (
            torch.from_numpy(
                state.astype(
                    np.uint8,
                    copy=False,
                )
            )
            .cpu()
            .contiguous()
        )

    try:
        return torch.tensor(
            state,
            dtype=torch.uint8,
            device="cpu",
        ).contiguous()

    except Exception as error:
        raise TypeError(
            f"{state_name} could not be converted "
            "to a CPU ByteTensor."
        ) from error


def capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),

        "torch_cpu": (
            torch.get_rng_state()
            .cpu()
            .contiguous()
        ),

        "torch_cuda": None,
    }

    if torch.cuda.is_available():
        state["torch_cuda"] = [
            cuda_state.cpu().contiguous()
            for cuda_state in (
                torch.cuda.get_rng_state_all()
            )
        ]

    return state


def restore_rng_state(
    state: dict,
):
    random.setstate(
        state["python"]
    )

    numpy_state = state["numpy"]

    if isinstance(
        numpy_state,
        list,
    ):
        numpy_state = tuple(
            numpy_state
        )

    np.random.set_state(
        numpy_state
    )

    torch.set_rng_state(
        as_cpu_byte_tensor(
            state["torch_cpu"],
            state_name=(
                "PyTorch CPU RNG state"
            ),
        )
    )

    saved_cuda_states = state.get(
        "torch_cuda"
    )

    if (
        torch.cuda.is_available()
        and saved_cuda_states is not None
    ):
        if isinstance(
            saved_cuda_states,
            torch.Tensor,
        ):
            saved_cuda_states = [
                saved_cuda_states
            ]

        normalized_states = [
            as_cpu_byte_tensor(
                cuda_state,
                state_name=(
                    f"CUDA RNG state {index}"
                ),
            )
            for index, cuda_state in enumerate(
                saved_cuda_states
            )
        ]

        current_gpu_count = (
            torch.cuda.device_count()
        )

        if (
            len(normalized_states)
            != current_gpu_count
        ):
            if (
                current_gpu_count == 1
                and len(
                    normalized_states
                ) >= 1
            ):
                normalized_states = [
                    normalized_states[0]
                ]

            else:
                raise RuntimeError(
                    "Saved CUDA RNG-state count does not "
                    "match the current GPU count."
                )

        torch.cuda.set_rng_state_all(
            normalized_states
        )


def optimizer_to_device(
    optimizer,
    target_device,
):
    for optimizer_state in (
        optimizer.state.values()
    ):
        for key, value in (
            optimizer_state.items()
        ):
            if isinstance(
                value,
                torch.Tensor,
            ):
                optimizer_state[key] = (
                    value.to(
                        target_device
                    )
                )


seed_everything(
    SCREENING_SEED
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.set_float32_matmul_precision(
        "high"
    )
except Exception:
    pass


# ============================================================
# 11. Hash and environment helpers
# ============================================================

def file_sha256(
    path: Path,
) -> str:
    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            hasher.update(block)

    return hasher.hexdigest()


def dictionary_sha256(
    dictionary: dict,
) -> str:
    encoded = json.dumps(
        dictionary,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def safe_package_version(
    package_name: str,
):
    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


# ============================================================
# 12. Verify the locked aggregation decision
# ============================================================

with open(
    LOCKED_AGGREGATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    locked_aggregation_record = (
        json.load(file)
    )

observed_locked_rule = (
    locked_aggregation_record.get(
        "locked_aggregation_rule"
    )
)

observed_primary_reference = (
    locked_aggregation_record.get(
        "primary_reference_rule"
    )
)

if (
    observed_locked_rule
    != LOCKED_AGGREGATION_RULE
):
    raise RuntimeError(
        "Notebook 03 did not lock the expected "
        "aggregation rule.\n"
        f"Expected: {LOCKED_AGGREGATION_RULE}\n"
        f"Observed: {observed_locked_rule}"
    )

if (
    observed_primary_reference
    != PRIMARY_REFERENCE_RULE
):
    raise RuntimeError(
        "Notebook 03 primary reference differs from "
        "the current protocol.\n"
        f"Expected: {PRIMARY_REFERENCE_RULE}\n"
        f"Observed: {observed_primary_reference}"
    )

print(
    "\nVerified locked aggregation:",
    observed_locked_rule,
)

print(
    "Verified primary reference:",
    observed_primary_reference,
)


# ============================================================
# 13. Freeze or validate the existing analysis plan
# ============================================================

immutable_analysis_plan = {
    "protocol_version": PROTOCOL_VERSION,

    "study_stage": (
        "Development-only Swin loss screening"
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,

    "model_family": "Swin Transformer",

    "resolved_model_name": (
        RESOLVED_SWIN_MODEL_NAME
    ),

    "screening_seed": int(
        SCREENING_SEED
    ),

    "primary_patient_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "locked_patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),

    "candidate_experiments": (
        EXPERIMENT_PRESETS
    ),

    "training_config": (
        BASE_TRAINING_CONFIG
    ),

    "checkpoint_selection_hierarchy": [
        "Validation patient QWK: higher",
        "Validation patient MAE: lower",
        "Validation patient macro F1: higher",
        "Validation image QWK: higher",
    ],

    "experiment_ranking_hierarchy": [
        "Pooled OOF patient QWK: higher",
        "Pooled OOF patient MAE: lower",
        "Pooled OOF patient macro F1: higher",
        "Pooled OOF image QWK: higher",
    ],

    "development_manifest_path": str(
        MANIFEST_PATH
    ),

    "development_manifest_sha256": (
        file_sha256(
            MANIFEST_PATH
        )
    ),

    "fold_assignment_path": str(
        FOLD_ASSIGNMENT_PATH
    ),

    "fold_assignment_sha256": (
        file_sha256(
            FOLD_ASSIGNMENT_PATH
        )
    ),

    "reference_audit_path": str(
        REFERENCE_AUDIT_PATH
    ),

    "reference_audit_sha256": (
        file_sha256(
            REFERENCE_AUDIT_PATH
        )
    ),

    "screening_plan_path": str(
        SCREENING_PLAN_PATH
    ),

    "screening_plan_sha256": (
        file_sha256(
            SCREENING_PLAN_PATH
        )
    ),

    "confirmation_plan_path": str(
        CONFIRMATION_PLAN_PATH
    ),

    "confirmation_plan_sha256": (
        file_sha256(
            CONFIRMATION_PLAN_PATH
        )
    ),

    "aggregation_plan_path": str(
        AGGREGATION_PLAN_PATH
    ),

    "aggregation_plan_sha256": (
        file_sha256(
            AGGREGATION_PLAN_PATH
        )
    ),

    "locked_aggregation_path": str(
        LOCKED_AGGREGATION_PATH
    ),

    "locked_aggregation_sha256": (
        file_sha256(
            LOCKED_AGGREGATION_PATH
        )
    ),

    "screening_interpretation": (
        "This notebook screens Swin loss formulations "
        "using development-only pooled OOF predictions. "
        "The top two candidates are shortlisted for "
        "multi-seed confirmation and are not considered "
        "final based on this single-seed screening."
    ),
}

analysis_plan_fingerprint = (
    dictionary_sha256(
        immutable_analysis_plan
    )
)

analysis_plan = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    **immutable_analysis_plan,
}

analysis_plan_path = (
    OUTPUT_ROOT
    / "analysis_plan_frozen_before_training.json"
)

if analysis_plan_path.exists():
    with open(
        analysis_plan_path,
        "r",
        encoding="utf-8",
    ) as file:
        existing_plan = json.load(file)

    if (
        existing_plan.get(
            "analysis_plan_fingerprint"
        )
        != analysis_plan_fingerprint
    ):
        raise RuntimeError(
            "The existing Swin analysis plan differs from "
            "the current scientific configuration.\n"
            "Do not mix protocols. Use a new versioned "
            "OUTPUT_ROOT."
        )

    print(
        "\nExisting Swin analysis plan validated:",
        analysis_plan_path,
    )

else:
    atomic_json_save(
        analysis_plan,
        analysis_plan_path,
    )

    print(
        "\nSaved frozen Swin analysis plan:",
        analysis_plan_path,
    )

print(
    "Analysis-plan fingerprint:",
    analysis_plan_fingerprint,
)


# ============================================================
# 14. Save runtime-only I/O recovery record
# ============================================================

runtime_io_recovery_record = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "reason": (
        "Transient Google Drive OSError Errno 5 during "
        "concurrent DataLoader worker image reads."
    ),

    "frozen_plan_num_workers": int(
        NUM_WORKERS
    ),

    "runtime_num_workers": int(
        RUNTIME_NUM_WORKERS
    ),

    "local_image_cache_enabled": bool(
        USE_LOCAL_IMAGE_CACHE
    ),

    "local_image_cache_root": str(
        LOCAL_IMAGE_CACHE_ROOT
    ),

    "image_read_max_retries": int(
        IMAGE_IO_MAX_RETRIES
    ),

    "image_read_initial_delay_seconds": float(
        IMAGE_IO_INITIAL_DELAY_SECONDS
    ),

    "scientific_configuration_changed": False,
    "model_architecture_changed": False,
    "loss_configuration_changed": False,
    "fold_assignment_changed": False,
    "patient_reference_changed": False,
    "patient_aggregation_changed": False,

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

atomic_json_save(
    runtime_io_recovery_record,
    OUTPUT_ROOT
    / "runtime_google_drive_io_recovery.json",
)


# ============================================================
# 15. Environment metadata
# ============================================================

environment_metadata = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "python": platform.python_version(),
    "platform": platform.platform(),

    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,

    "numpy": np.__version__,
    "pandas": pd.__version__,

    "scikit_learn": (
        safe_package_version(
            "scikit-learn"
        )
    ),

    "pillow": (
        safe_package_version(
            "Pillow"
        )
    ),

    "cuda_available": bool(
        torch.cuda.is_available()
    ),

    "cuda_version": torch.version.cuda,

    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "resolved_model_name": (
        RESOLVED_SWIN_MODEL_NAME
    ),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "runtime_num_workers": int(
        RUNTIME_NUM_WORKERS
    ),

    "local_image_cache_enabled": bool(
        USE_LOCAL_IMAGE_CACHE
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

atomic_json_save(
    environment_metadata,
    OUTPUT_ROOT
    / "environment_metadata.json",
)


# ============================================================
# 16. Load and validate frozen development manifest
# ============================================================

trainval_df = pd.read_csv(
    MANIFEST_PATH
)

required_manifest_columns = {
    "relative_path",
    "filename",
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
}

missing_manifest_columns = (
    required_manifest_columns
    - set(trainval_df.columns)
)

if missing_manifest_columns:
    raise RuntimeError(
        "Development manifest is missing columns: "
        f"{sorted(missing_manifest_columns)}"
    )

trainval_df["path"] = (
    trainval_df["relative_path"]
    .map(
        lambda relative_path: str(
            DEVELOPMENT_ROOT
            / Path(relative_path)
        )
    )
)

for integer_column in [
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
]:
    trainval_df[integer_column] = (
        trainval_df[integer_column]
        .astype(int)
    )

trainval_df = (
    trainval_df
    .sort_values(
        [
            "patient_id",
            "relative_path",
        ]
    )
    .reset_index(drop=True)
)

if len(trainval_df) != 9590:
    raise RuntimeError(
        f"Expected 9,590 development images; "
        f"found {len(trainval_df)}."
    )

if (
    trainval_df[
        "patient_id"
    ].nunique()
    != 478
):
    raise RuntimeError(
        "Expected 478 development patients; "
        f"found {trainval_df['patient_id'].nunique()}."
    )

if trainval_df[
    "path"
].duplicated().any():
    raise RuntimeError(
        "Duplicate image paths exist in the manifest."
    )

observed_folds = set(
    trainval_df["cv_fold"].unique()
)

expected_folds = set(
    range(NUM_FOLDS)
)

if observed_folds != expected_folds:
    raise RuntimeError(
        f"Expected folds {sorted(expected_folds)}; "
        f"found {sorted(observed_folds)}."
    )


# ============================================================
# 17. Validate frozen folds and references
# ============================================================

fold_assignment_df = pd.read_csv(
    FOLD_ASSIGNMENT_PATH
)

reference_audit_df = pd.read_csv(
    REFERENCE_AUDIT_PATH
)

fold_assignment_df["patient_id"] = (
    fold_assignment_df["patient_id"]
    .astype(int)
)

fold_assignment_df["patient_label"] = (
    fold_assignment_df["patient_label"]
    .astype(int)
)

fold_assignment_df["cv_fold"] = (
    fold_assignment_df["cv_fold"]
    .astype(int)
)

reference_audit_df["patient_id"] = (
    reference_audit_df["patient_id"]
    .astype(int)
)

reference_audit_df[
    PRIMARY_REFERENCE_RULE
] = (
    reference_audit_df[
        PRIMARY_REFERENCE_RULE
    ]
    .astype(int)
)

reference_check_df = (
    fold_assignment_df[
        [
            "patient_id",
            "patient_label",
            "cv_fold",
        ]
    ]
    .merge(
        reference_audit_df[
            [
                "patient_id",
                PRIMARY_REFERENCE_RULE,
            ]
        ],
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)

if len(reference_check_df) != 478:
    raise RuntimeError(
        "Reference validation did not include "
        "all 478 patients."
    )

if not (
    reference_check_df["patient_label"]
    ==
    reference_check_df[
        PRIMARY_REFERENCE_RULE
    ]
).all():
    raise RuntimeError(
        "Frozen patient labels do not match modal_low."
    )

manifest_patient_check_df = (
    trainval_df[
        [
            "patient_id",
            "cv_fold",
            "patient_y_true",
        ]
    ]
    .drop_duplicates()
    .merge(
        fold_assignment_df[
            [
                "patient_id",
                "cv_fold",
                "patient_label",
            ]
        ],
        on="patient_id",
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_manifest",
            "_frozen",
        ),
    )
)

if len(
    manifest_patient_check_df
) != 478:
    raise RuntimeError(
        "Manifest/fold validation did not include "
        "all patients."
    )

if not (
    manifest_patient_check_df[
        "cv_fold_manifest"
    ]
    ==
    manifest_patient_check_df[
        "cv_fold_frozen"
    ]
).all():
    raise RuntimeError(
        "Manifest folds differ from frozen folds."
    )

if not (
    manifest_patient_check_df[
        "patient_y_true"
    ]
    ==
    manifest_patient_check_df[
        "patient_label"
    ]
).all():
    raise RuntimeError(
        "Manifest references differ from frozen "
        "modal-low labels."
    )

print(
    "\nDevelopment images:",
    len(trainval_df),
)

print(
    "Development patients:",
    trainval_df[
        "patient_id"
    ].nunique(),
)

print(
    "Frozen folds:",
    [
        int(value)
        for value in sorted(
            observed_folds
        )
    ],
)

print(
    "Primary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Locked patient aggregation:",
    LOCKED_AGGREGATION_RULE,
)


# ============================================================
# 18. Robust Google Drive image reading and local caching
# ============================================================

def local_cache_path_for_row(
    relative_path,
) -> Path:
    return (
        LOCAL_IMAGE_CACHE_ROOT
        / Path(str(relative_path))
    )


def read_source_bytes_with_retry(
    source_path,
):
    source_path = str(
        source_path
    )

    last_error = None

    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):
        try:
            with open(
                source_path,
                "rb",
            ) as source_file:
                image_bytes = (
                    source_file.read()
                )

            if len(image_bytes) == 0:
                raise OSError(
                    "Image file was read as zero bytes."
                )

            return image_bytes

        except (
            OSError,
            IOError,
        ) as error:
            last_error = error

            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES - 1
            ):
                break

            delay = retry_delay(
                attempt_index,
                IMAGE_IO_INITIAL_DELAY_SECONDS,
                IMAGE_IO_MAX_DELAY_SECONDS,
            )

            print(
                "\nTransient Google Drive image-read failure."
            )

            print(
                f"Attempt {attempt_index + 1}/"
                f"{IMAGE_IO_MAX_RETRIES}"
            )

            print(
                "Image:",
                source_path,
            )

            print(
                "Error:",
                repr(error),
            )

            print(
                f"Retrying in {delay:.1f} seconds..."
            )

            time.sleep(delay)

    raise OSError(
        "Image could not be read from Google Drive "
        f"after {IMAGE_IO_MAX_RETRIES} attempts:\n"
        f"{source_path}\n"
        f"Last error: {repr(last_error)}"
    )


def ensure_local_cached_image(
    source_path,
    relative_path,
) -> Path:
    source_path = str(
        source_path
    )

    local_path = local_cache_path_for_row(
        relative_path
    )

    if (
        local_path.exists()
        and local_path.stat().st_size > 0
    ):
        return local_path

    local_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    image_bytes = (
        read_source_bytes_with_retry(
            source_path
        )
    )

    temp_path = temporary_path(
        local_path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "wb",
    ) as local_file:
        local_file.write(
            image_bytes
        )

    os.replace(
        temp_path,
        local_path,
    )

    return local_path


def open_rgb_image_with_retry(
    source_path,
    relative_path,
):
    source_path = str(
        source_path
    )

    last_error = None

    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):
        local_path = None

        try:
            if USE_LOCAL_IMAGE_CACHE:
                local_path = (
                    ensure_local_cached_image(
                        source_path=source_path,
                        relative_path=relative_path,
                    )
                )

                read_path = local_path

            else:
                read_path = Path(
                    source_path
                )

            with open(
                read_path,
                "rb",
            ) as image_file:
                image_bytes = (
                    image_file.read()
                )

            if len(image_bytes) == 0:
                raise OSError(
                    "Image was read as zero bytes."
                )

            with Image.open(
                io.BytesIO(
                    image_bytes
                )
            ) as image:
                image.load()

                rgb_image = image.convert(
                    "RGB"
                )

            return rgb_image

        except (
            OSError,
            IOError,
            ValueError,
        ) as error:
            last_error = error

            # A partially cached or undecodable local copy is
            # removed so the next attempt copies it again.
            if (
                local_path is not None
                and local_path.exists()
            ):
                try:
                    local_path.unlink()
                except Exception:
                    pass

            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES - 1
            ):
                break

            delay = retry_delay(
                attempt_index,
                IMAGE_IO_INITIAL_DELAY_SECONDS,
                IMAGE_IO_MAX_DELAY_SECONDS,
            )

            print(
                "\nTransient image open/decode failure."
            )

            print(
                f"Attempt {attempt_index + 1}/"
                f"{IMAGE_IO_MAX_RETRIES}"
            )

            print(
                "Image:",
                source_path,
            )

            print(
                "Error:",
                repr(error),
            )

            print(
                f"Retrying in {delay:.1f} seconds..."
            )

            time.sleep(delay)

    raise OSError(
        "Image could not be opened after "
        f"{IMAGE_IO_MAX_RETRIES} attempts:\n"
        f"{source_path}\n"
        f"Last error: {repr(last_error)}"
    )


# ============================================================
# 19. Dataset and transforms
# ============================================================

class LIMUCDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform=None,
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(
        self,
        index,
    ):
        row = self.df.iloc[index]

        canonical_path = str(
            row["path"]
        )

        relative_path = str(
            row["relative_path"]
        )

        image = open_rgb_image_with_retry(
            source_path=canonical_path,
            relative_path=relative_path,
        )

        if self.transform is not None:
            image = self.transform(
                image
            )

        return {
            "image": image,

            "label": torch.tensor(
                int(row["label"]),
                dtype=torch.long,
            ),

            "patient_id": torch.tensor(
                int(row["patient_id"]),
                dtype=torch.long,
            ),

            "patient_y_true": torch.tensor(
                int(
                    row[
                        "patient_y_true"
                    ]
                ),
                dtype=torch.long,
            ),

            # Keep canonical Drive path in predictions.
            "path": canonical_path,
        }


def build_train_transform():
    return transforms.Compose([
        transforms.RandomResizedCrop(
            size=(
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),

            scale=(
                0.75,
                1.00,
            ),

            ratio=(
                0.90,
                1.10,
            ),
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomRotation(
            degrees=8
        ),

        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
            hue=0.02,
        ),

        transforms.GaussianBlur(
            kernel_size=3,

            sigma=(
                0.1,
                1.2,
            ),
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


def build_eval_transform():
    return transforms.Compose([
        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


# ============================================================
# 20. Metrics and locked patient aggregation
# ============================================================

PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]


def compute_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    if len(y_true) == 0:
        raise ValueError(
            "Cannot calculate metrics on an empty set."
        )

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred have different lengths."
        )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,
        labels=[
            0,
            1,
            2,
            3,
        ],
        weights="quadratic",
    )

    return {
        "qwk": (
            float(qwk)
            if np.isfinite(qwk)
            else np.nan
        ),

        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),

        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=[
                    0,
                    1,
                    2,
                    3,
                ],
                average="macro",
                zero_division=0,
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
    }


def make_confusion_df(
    y_true,
    y_pred,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1,
            2,
            3,
        ],
    )

    return pd.DataFrame(
        matrix,

        index=[
            "true_0",
            "true_1",
            "true_2",
            "true_3",
        ],

        columns=[
            "pred_0",
            "pred_1",
            "pred_2",
            "pred_3",
        ],
    )


def aggregate_mean_probabilities(
    image_prediction_df: pd.DataFrame,
):
    if (
        LOCKED_AGGREGATION_RULE
        != "mean_probability_argmax"
    ):
        raise RuntimeError(
            "This function implements only the locked "
            "mean_probability_argmax rule."
        )

    required_columns = {
        "patient_id",
        "patient_y_true",
        "path",
        *PROBABILITY_COLUMNS,
    }

    missing_columns = (
        required_columns
        - set(
            image_prediction_df.columns
        )
    )

    if missing_columns:
        raise RuntimeError(
            "Prediction dataframe is missing columns: "
            f"{sorted(missing_columns)}"
        )

    patient_reference_counts = (
        image_prediction_df
        .groupby("patient_id")[
            "patient_y_true"
        ]
        .nunique()
    )

    if (
        patient_reference_counts.max()
        != 1
    ):
        raise RuntimeError(
            "A patient has inconsistent reference labels."
        )

    patient_probability_df = (
        image_prediction_df
        .groupby(
            "patient_id",
            as_index=False,
        )[PROBABILITY_COLUMNS]
        .mean()
    )

    patient_metadata_df = (
        image_prediction_df
        .groupby(
            "patient_id",
            as_index=False,
        )
        .agg(
            y_true=(
                "patient_y_true",
                "first",
            ),

            n_images=(
                "path",
                "size",
            ),
        )
    )

    patient_df = (
        patient_metadata_df
        .merge(
            patient_probability_df,
            on="patient_id",
            how="inner",
            validate="one_to_one",
        )
    )

    patient_df["patient_id"] = (
        patient_df["patient_id"]
        .astype(int)
    )

    patient_df["y_true"] = (
        patient_df["y_true"]
        .astype(int)
    )

    patient_df["n_images"] = (
        patient_df["n_images"]
        .astype(int)
    )

    patient_df["y_pred"] = (
        patient_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy()
        .argmax(axis=1)
        .astype(int)
    )

    return (
        patient_df
        .sort_values("patient_id")
        .reset_index(drop=True)
    )


# ============================================================
# 21. Prediction validation
# ============================================================

def validate_image_predictions(
    prediction_df: pd.DataFrame,
    expected_df: pd.DataFrame,
    expected_seed=None,
    expected_fold=None,
):
    prediction_df = prediction_df.copy()

    required_columns = {
        "path",
        "patient_id",
        "y_true",
        "patient_y_true",
        "y_pred",
        *PROBABILITY_COLUMNS,
    }

    missing_columns = (
        required_columns
        - set(prediction_df.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Prediction file is missing columns: "
            f"{sorted(missing_columns)}"
        )

    prediction_df["path"] = (
        prediction_df["path"]
        .astype(str)
    )

    for integer_column in [
        "patient_id",
        "y_true",
        "patient_y_true",
        "y_pred",
    ]:
        prediction_df[
            integer_column
        ] = (
            prediction_df[
                integer_column
            ]
            .astype(int)
        )

    if prediction_df[
        "path"
    ].duplicated().any():
        raise RuntimeError(
            "Duplicate paths exist in a prediction file."
        )

    if len(prediction_df) != len(
        expected_df
    ):
        raise RuntimeError(
            "Prediction row count differs from expected.\n"
            f"Expected: {len(expected_df)}\n"
            f"Observed: {len(prediction_df)}"
        )

    probability_array = (
        prediction_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )

    if not np.isfinite(
        probability_array
    ).all():
        raise RuntimeError(
            "Prediction probabilities contain "
            "NaN or infinity."
        )

    probability_sums = (
        probability_array.sum(axis=1)
    )

    if not np.allclose(
        probability_sums,
        1.0,
        atol=2e-3,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Prediction probability rows do not sum to one."
        )

    prediction_df["y_pred"] = (
        probability_array
        .argmax(axis=1)
        .astype(int)
    )

    expected_metadata = (
        expected_df[
            [
                "path",
                "patient_id",
                "label",
                "patient_y_true",
            ]
        ]
        .rename(
            columns={
                "patient_id": (
                    "patient_id_expected"
                ),

                "label": (
                    "y_true_expected"
                ),

                "patient_y_true": (
                    "patient_y_true_expected"
                ),
            }
        )
    )

    validation_df = (
        prediction_df[
            [
                "path",
                "patient_id",
                "y_true",
                "patient_y_true",
            ]
        ]
        .merge(
            expected_metadata,
            on="path",
            how="inner",
            validate="one_to_one",
        )
    )

    if len(validation_df) != len(
        expected_df
    ):
        raise RuntimeError(
            "Prediction paths do not match expected images."
        )

    if not (
        validation_df["patient_id"]
        ==
        validation_df[
            "patient_id_expected"
        ]
    ).all():
        raise RuntimeError(
            "Prediction patient IDs differ from the manifest."
        )

    if not (
        validation_df["y_true"]
        ==
        validation_df[
            "y_true_expected"
        ]
    ).all():
        raise RuntimeError(
            "Prediction image labels differ from the manifest."
        )

    if not (
        validation_df[
            "patient_y_true"
        ]
        ==
        validation_df[
            "patient_y_true_expected"
        ]
    ).all():
        raise RuntimeError(
            "Prediction patient references differ from "
            "the frozen reference."
        )

    if expected_seed is not None:
        prediction_df[
            "training_seed"
        ] = int(expected_seed)

    if expected_fold is not None:
        prediction_df[
            "fold"
        ] = int(expected_fold)

    return (
        prediction_df
        .sort_values("path")
        .reset_index(drop=True)
    )


# ============================================================
# 22. Ordinal losses
# ============================================================

class ExpectedValueOrdinalMAELoss(nn.Module):
    def __init__(
        self,
        num_classes: int,
    ):
        super().__init__()

        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        expected_class = (
            probabilities
            *
            self.class_values.unsqueeze(0)
        ).sum(dim=1)

        return torch.abs(
            expected_class
            - targets.float()
        ).mean()


class ExpectedOrdinalDistanceLoss(nn.Module):
    def __init__(
        self,
        num_classes: int,
        alpha: float,
    ):
        super().__init__()

        self.alpha = float(alpha)

        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        targets = (
            targets.float()
            .unsqueeze(1)
        )

        distances = torch.abs(
            targets
            -
            self.class_values.unsqueeze(0)
        ) ** self.alpha

        return (
            probabilities
            * distances
        ).sum(dim=1).mean()


class CombinedLoss(nn.Module):
    def __init__(
        self,
        loss_type: str,
        ce_weight,
        ordinal_loss_weight: float,
        ordinal_alpha: float,
    ):
        super().__init__()

        valid_loss_types = {
            "ce",
            "ce_expected_value_mae",
            "ce_expected_distance",
        }

        if loss_type not in valid_loss_types:
            raise ValueError(
                f"Unsupported loss type: {loss_type}"
            )

        self.loss_type = loss_type

        self.ordinal_loss_weight = float(
            ordinal_loss_weight
        )

        self.cross_entropy = nn.CrossEntropyLoss(
            weight=ce_weight,
            label_smoothing=LABEL_SMOOTHING,
        )

        if loss_type == "ce":
            self.ordinal_loss = None

        elif (
            loss_type
            == "ce_expected_value_mae"
        ):
            self.ordinal_loss = (
                ExpectedValueOrdinalMAELoss(
                    num_classes=NUM_CLASSES
                )
            )

        else:
            self.ordinal_loss = (
                ExpectedOrdinalDistanceLoss(
                    num_classes=NUM_CLASSES,
                    alpha=ordinal_alpha,
                )
            )

    def forward(
        self,
        logits,
        targets,
    ):
        ce_loss = self.cross_entropy(
            logits,
            targets,
        )

        if self.ordinal_loss is None:
            ordinal_loss = torch.zeros(
                (),
                dtype=logits.dtype,
                device=logits.device,
            )

            total_loss = ce_loss

        else:
            ordinal_loss = self.ordinal_loss(
                logits,
                targets,
            )

            total_loss = (
                ce_loss
                +
                self.ordinal_loss_weight
                * ordinal_loss
            )

        return (
            total_loss,
            ce_loss.detach(),
            ordinal_loss.detach(),
        )


# ============================================================
# 23. Scheduler and mixed precision helpers
# ============================================================

def build_scheduler(
    optimizer,
):
    def lr_lambda(
        epoch,
    ):
        if epoch < WARMUP_EPOCHS:
            return (
                float(epoch + 1)
                /
                float(
                    max(
                        1,
                        WARMUP_EPOCHS,
                    )
                )
            )

        progress = (
            float(
                epoch
                - WARMUP_EPOCHS
            )
            /
            float(
                max(
                    1,
                    EPOCHS
                    - WARMUP_EPOCHS,
                )
            )
        )

        cosine_value = 0.5 * (
            1.0
            +
            math.cos(
                math.pi
                * progress
            )
        )

        return max(
            MIN_LR_RATIO,
            cosine_value,
        )

    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda,
    )


def create_grad_scaler(
    enabled: bool,
):
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=enabled,
        )

    except Exception:
        return torch.cuda.amp.GradScaler(
            enabled=enabled
        )


def autocast_context(
    enabled: bool,
):
    try:
        return torch.amp.autocast(
            device_type="cuda",
            enabled=enabled,
        )

    except Exception:
        return torch.cuda.amp.autocast(
            enabled=enabled
        )


# ============================================================
# 24. Training helpers
# ============================================================

def calculate_class_weights(
    training_df: pd.DataFrame,
    device,
):
    class_counts = (
        training_df["label"]
        .value_counts()
        .reindex(
            range(NUM_CLASSES),
            fill_value=0,
        )
        .sort_index()
    )

    if (
        class_counts == 0
    ).any():
        missing_classes = (
            class_counts[
                class_counts == 0
            ]
            .index
            .tolist()
        )

        raise RuntimeError(
            "Training fold is missing classes: "
            f"{missing_classes}"
        )

    total_images = int(
        class_counts.sum()
    )

    class_weights = (
        total_images
        /
        (
            NUM_CLASSES
            * class_counts
        )
    )

    class_weight_tensor = torch.tensor(
        class_weights.to_numpy(),
        dtype=torch.float32,
        device=device,
    )

    return (
        class_weight_tensor,

        {
            str(key): int(value)
            for key, value in (
                class_counts.items()
            )
        },

        {
            str(key): float(value)
            for key, value in (
                class_weights.items()
            )
        },
    )


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
):
    model.train()

    total_loss = 0.0

    y_true = []
    y_pred = []

    use_amp = (
        USE_MIXED_PRECISION
        and device.type == "cuda"
        and scaler.is_enabled()
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    number_of_batches = len(
        loader
    )

    for batch_index, batch in enumerate(
        loader
    ):
        images = batch[
            "image"
        ].to(
            device,
            non_blocking=True,
        )

        labels = batch[
            "label"
        ].to(
            device,
            non_blocking=True,
        )

        group_start = (
            batch_index
            //
            GRADIENT_ACCUMULATION_STEPS
        ) * GRADIENT_ACCUMULATION_STEPS

        accumulation_group_size = min(
            GRADIENT_ACCUMULATION_STEPS,

            number_of_batches
            - group_start,
        )

        with autocast_context(
            enabled=use_amp
        ):
            logits = model(
                images
            )

            (
                raw_loss,
                _,
                _,
            ) = criterion(
                logits,
                labels,
            )

            scaled_loss = (
                raw_loss
                /
                accumulation_group_size
            )

        if use_amp:
            scaler.scale(
                scaled_loss
            ).backward()

        else:
            scaled_loss.backward()

        should_step = (
            (
                batch_index + 1
            )
            %
            GRADIENT_ACCUMULATION_STEPS
            == 0
            or
            (
                batch_index + 1
            )
            == number_of_batches
        )

        if should_step:
            if use_amp:
                scaler.unscale_(
                    optimizer
                )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP_NORM,
            )

            if use_amp:
                scaler.step(
                    optimizer
                )

                scaler.update()

            else:
                optimizer.step()

            optimizer.zero_grad(
                set_to_none=True
            )

        total_loss += (
            float(raw_loss.item())
            * images.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        y_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    metrics = compute_metrics(
        y_true,
        y_pred,
    )

    metrics["loss"] = (
        total_loss
        /
        len(loader.dataset)
    )

    return metrics


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    total_loss = 0.0

    image_y_true = []
    image_y_pred = []
    image_probabilities = []

    patient_ids = []
    patient_y_true = []
    image_paths = []

    use_amp = (
        USE_MIXED_PRECISION
        and device.type == "cuda"
    )

    for batch in loader:
        images = batch[
            "image"
        ].to(
            device,
            non_blocking=True,
        )

        labels = batch[
            "label"
        ].to(
            device,
            non_blocking=True,
        )

        with autocast_context(
            enabled=use_amp
        ):
            logits = model(
                images
            )

            (
                loss,
                _,
                _,
            ) = criterion(
                logits,
                labels,
            )

            probability_list = [
                torch.softmax(
                    logits,
                    dim=1,
                )
            ]

            if (
                USE_TTA
                and TTA_HORIZONTAL_FLIP
            ):
                flipped_images = torch.flip(
                    images,
                    dims=[3],
                )

                flipped_logits = model(
                    flipped_images
                )

                probability_list.append(
                    torch.softmax(
                        flipped_logits,
                        dim=1,
                    )
                )

            probabilities = torch.stack(
                probability_list,
                dim=0,
            ).mean(dim=0)

        predictions = probabilities.argmax(
            dim=1
        )

        total_loss += (
            float(loss.item())
            * images.size(0)
        )

        image_y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        image_y_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        image_probabilities.extend(
            probabilities.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        patient_ids.extend(
            batch["patient_id"]
            .cpu()
            .numpy()
            .tolist()
        )

        patient_y_true.extend(
            batch["patient_y_true"]
            .cpu()
            .numpy()
            .tolist()
        )

        image_paths.extend(
            batch["path"]
        )

    image_metrics = compute_metrics(
        image_y_true,
        image_y_pred,
    )

    image_metrics["loss"] = (
        total_loss
        /
        len(loader.dataset)
    )

    prediction_df = pd.DataFrame({
        "path": image_paths,
        "patient_id": patient_ids,
        "y_true": image_y_true,
        "patient_y_true": patient_y_true,
        "y_pred": image_y_pred,
    })

    probability_df = pd.DataFrame(
        image_probabilities,
        columns=PROBABILITY_COLUMNS,
    )

    prediction_df = pd.concat(
        [
            prediction_df,
            probability_df,
        ],
        axis=1,
    )

    patient_df = (
        aggregate_mean_probabilities(
            prediction_df
        )
    )

    patient_metrics = compute_metrics(
        patient_df["y_true"],
        patient_df["y_pred"],
    )

    return (
        image_metrics,
        patient_metrics,
        prediction_df,
        patient_df,
    )


# ============================================================
# 25. Patient-primary checkpoint comparison
# ============================================================

def safe_metric_for_max(
    value,
):
    if (
        value is None
        or not np.isfinite(value)
    ):
        return -np.inf

    return float(value)


def safe_metric_for_min(
    value,
):
    if (
        value is None
        or not np.isfinite(value)
    ):
        return np.inf

    return float(value)


def is_better_checkpoint(
    candidate_patient_metrics,
    best_patient_metrics,
    candidate_image_metrics,
    best_image_metrics,
):
    if best_patient_metrics is None:
        return True

    candidate_tuple = (
        safe_metric_for_max(
            candidate_patient_metrics[
                "qwk"
            ]
        ),

        -safe_metric_for_min(
            candidate_patient_metrics[
                "mae"
            ]
        ),

        safe_metric_for_max(
            candidate_patient_metrics[
                "macro_f1"
            ]
        ),

        safe_metric_for_max(
            candidate_image_metrics[
                "qwk"
            ]
        ),
    )

    best_tuple = (
        safe_metric_for_max(
            best_patient_metrics[
                "qwk"
            ]
        ),

        -safe_metric_for_min(
            best_patient_metrics[
                "mae"
            ]
        ),

        safe_metric_for_max(
            best_patient_metrics[
                "macro_f1"
            ]
        ),

        safe_metric_for_max(
            best_image_metrics[
                "qwk"
            ]
        ),
    )

    return candidate_tuple > best_tuple


# ============================================================
# 26. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "\nDevice:",
    device,
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )


# ============================================================
# 27. Completed-fold loader
# ============================================================

def load_completed_fold(
    fold_directory: Path,
    expected_fold_fingerprint: str,
    expected_validation_df: pd.DataFrame,
):
    fold_result_path = (
        fold_directory
        / "fold_result.json"
    )

    image_prediction_path = (
        fold_directory
        / "validation_image_predictions.csv"
    )

    patient_prediction_path = (
        fold_directory
        / "validation_patient_predictions.csv"
    )

    best_checkpoint_path = (
        fold_directory
        / "best_checkpoint.pt"
    )

    required_paths = [
        fold_result_path,
        image_prediction_path,
        patient_prediction_path,
        best_checkpoint_path,
    ]

    if not all(
        path.exists()
        for path in required_paths
    ):
        return None

    with open(
        fold_result_path,
        "r",
        encoding="utf-8",
    ) as file:
        fold_result = json.load(file)

    if (
        fold_result.get(
            "fold_fingerprint"
        )
        != expected_fold_fingerprint
    ):
        raise RuntimeError(
            "A completed fold exists, but its configuration "
            "does not match the current frozen plan:\n"
            f"{fold_directory}"
        )

    image_prediction_df = pd.read_csv(
        image_prediction_path
    )

    image_prediction_df = (
        validate_image_predictions(
            prediction_df=(
                image_prediction_df
            ),

            expected_df=(
                expected_validation_df
            ),

            expected_seed=int(
                fold_result[
                    "training_seed"
                ]
            ),

            expected_fold=int(
                fold_result["fold"]
            ),
        )
    )

    patient_prediction_df = (
        aggregate_mean_probabilities(
            image_prediction_df
        )
    )

    patient_prediction_df["fold"] = int(
        fold_result["fold"]
    )

    patient_prediction_df[
        "training_seed"
    ] = int(
        fold_result["training_seed"]
    )

    print(
        "\nLoaded completed fold:",
        fold_result[
            "experiment_name"
        ],
        "| seed",
        fold_result[
            "training_seed"
        ],
        "| fold",
        fold_result["fold"],
    )

    return {
        "fold_result": fold_result,
        "image_predictions": (
            image_prediction_df
        ),
        "patient_predictions": (
            patient_prediction_df
        ),
    }


# ============================================================
# 28. Run or resume one fold
# ============================================================

def run_or_resume_fold(
    experiment_name: str,
    experiment_config: dict,
    training_seed: int,
    fold: int,
    experiment_directory: Path,
):
    fold_seed = (
        int(training_seed)
        + int(fold)
    )

    fold_directory = (
        experiment_directory
        / f"fold_{fold}"
    )

    fold_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    # This structure is kept identical to the earlier code so
    # existing fold fingerprints and checkpoints remain valid.
    fold_configuration = {
        "protocol_version": (
            PROTOCOL_VERSION
        ),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "experiment_name": (
            experiment_name
        ),

        "experiment_config": (
            experiment_config
        ),

        "training_seed": int(
            training_seed
        ),

        "fold": int(fold),

        "fold_seed": int(
            fold_seed
        ),
    }

    fold_fingerprint = (
        dictionary_sha256(
            fold_configuration
        )
    )

    training_df = (
        trainval_df[
            trainval_df["cv_fold"]
            != fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    validation_df = (
        trainval_df[
            trainval_df["cv_fold"]
            == fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    patient_overlap = (
        set(
            training_df[
                "patient_id"
            ].unique()
        )
        &
        set(
            validation_df[
                "patient_id"
            ].unique()
        )
    )

    if patient_overlap:
        raise RuntimeError(
            f"Patient leakage detected in fold {fold}."
        )

    if RESUME_COMPLETED_FOLDS:
        completed_fold = load_completed_fold(
            fold_directory=fold_directory,

            expected_fold_fingerprint=(
                fold_fingerprint
            ),

            expected_validation_df=(
                validation_df
            ),
        )

        if completed_fold is not None:
            return completed_fold

    print(
        "\n"
        + "=" * 105
    )

    print(
        f"{experiment_name} | "
        f"seed={training_seed} | "
        f"fold={fold + 1}/{NUM_FOLDS}"
    )

    print("=" * 105)

    print(
        "Training images:",
        len(training_df),
    )

    print(
        "Validation images:",
        len(validation_df),
    )

    print(
        "Training patients:",
        training_df[
            "patient_id"
        ].nunique(),
    )

    print(
        "Validation patients:",
        validation_df[
            "patient_id"
        ].nunique(),
    )

    validation_patient_distribution = (
        validation_df[
            [
                "patient_id",
                "patient_y_true",
            ]
        ]
        .drop_duplicates()
        ["patient_y_true"]
        .value_counts()
        .reindex(
            range(NUM_CLASSES),
            fill_value=0,
        )
        .sort_index()
    )

    for mayo_class in range(
        NUM_CLASSES
    ):
        print(
            f"Validation Mayo {mayo_class} patients:",
            int(
                validation_patient_distribution.loc[
                    mayo_class
                ]
            ),
        )

    training_dataset = LIMUCDataset(
        training_df,
        transform=build_train_transform(),
    )

    validation_dataset = LIMUCDataset(
        validation_df,
        transform=build_eval_transform(),
    )

    training_generator = torch.Generator()
    validation_generator = torch.Generator()

    training_generator.manual_seed(
        fold_seed
    )

    validation_generator.manual_seed(
        fold_seed + 100000
    )

    # Runtime workers are zero to avoid concurrent Drive reads.
    training_loader = DataLoader(
        training_dataset,

        batch_size=TRAIN_BATCH_SIZE,

        shuffle=True,

        num_workers=(
            RUNTIME_NUM_WORKERS
        ),

        pin_memory=(
            device.type == "cuda"
        ),

        drop_last=False,

        worker_init_fn=seed_worker,

        generator=training_generator,

        persistent_workers=False,
    )

    validation_loader = DataLoader(
        validation_dataset,

        batch_size=VALIDATION_BATCH_SIZE,

        shuffle=False,

        num_workers=(
            RUNTIME_NUM_WORKERS
        ),

        pin_memory=(
            device.type == "cuda"
        ),

        drop_last=False,

        worker_init_fn=seed_worker,

        generator=validation_generator,

        persistent_workers=False,
    )

    last_checkpoint_path = (
        fold_directory
        / "last_epoch_checkpoint.pt"
    )

    best_checkpoint_path = (
        fold_directory
        / "best_checkpoint.pt"
    )

    history_path = (
        fold_directory
        / "history.csv"
    )

    image_prediction_path = (
        fold_directory
        / "validation_image_predictions.csv"
    )

    patient_prediction_path = (
        fold_directory
        / "validation_patient_predictions.csv"
    )

    fold_result_path = (
        fold_directory
        / "fold_result.json"
    )

    resume_payload = None

    if (
        RESUME_INTERRUPTED_FOLDS
        and last_checkpoint_path.exists()
    ):
        try:
            resume_payload = safe_torch_load(
                last_checkpoint_path,
                map_location="cpu",
            )

            if (
                resume_payload.get(
                    "fold_fingerprint"
                )
                != fold_fingerprint
            ):
                raise RuntimeError(
                    "Interrupted checkpoint belongs to "
                    "another configuration."
                )

            print(
                "\nFound resumable checkpoint:",
                last_checkpoint_path,
            )

        except Exception as error:
            corrupt_path = (
                last_checkpoint_path
                .with_name(
                    "last_epoch_checkpoint_corrupt_"
                    +
                    datetime.now(
                        timezone.utc
                    ).strftime(
                        "%Y%m%d_%H%M%S"
                    )
                    +
                    ".pt"
                )
            )

            os.replace(
                last_checkpoint_path,
                corrupt_path,
            )

            print(
                "\nThe resume checkpoint could not be loaded."
            )

            print(
                "It was moved to:",
                corrupt_path,
            )

            print(
                "This fold will restart from epoch 1."
            )

            print(
                "Loading error:",
                repr(error),
            )

            resume_payload = None

    seed_everything(
        fold_seed
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = timm.create_model(
        RESOLVED_SWIN_MODEL_NAME,

        pretrained=(
            True
            if resume_payload is None
            else False
        ),

        num_classes=NUM_CLASSES,

        drop_rate=DROP_RATE,

        drop_path_rate=DROP_PATH_RATE,
    ).to(device)

    pretrained_cfg = copy.deepcopy(
        getattr(
            model,
            "pretrained_cfg",
            {},
        )
    )

    parameter_count = int(
        sum(
            parameter.numel()
            for parameter in (
                model.parameters()
            )
        )
    )

    (
        class_weight_tensor,
        training_class_counts,
        training_class_weights,
    ) = calculate_class_weights(
        training_df=training_df,
        device=device,
    )

    criterion = CombinedLoss(
        loss_type=(
            experiment_config[
                "loss_type"
            ]
        ),

        ce_weight=(
            class_weight_tensor
            if USE_CLASS_WEIGHTS_IN_CE
            else None
        ),

        ordinal_loss_weight=(
            experiment_config[
                "ordinal_loss_weight"
            ]
        ),

        ordinal_alpha=(
            experiment_config[
                "ordinal_alpha"
            ]
        ),
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY,
    )

    scheduler = build_scheduler(
        optimizer
    )

    scaler = create_grad_scaler(
        enabled=(
            USE_MIXED_PRECISION
            and device.type == "cuda"
        )
    )

    start_epoch = 1
    best_epoch = -1

    best_patient_metrics = None
    best_image_metrics = None

    patience_counter = 0
    history = []

    if resume_payload is not None:
        model.load_state_dict(
            resume_payload[
                "model_state_dict"
            ]
        )

        optimizer.load_state_dict(
            resume_payload[
                "optimizer_state_dict"
            ]
        )

        optimizer_to_device(
            optimizer,
            device,
        )

        scheduler.load_state_dict(
            resume_payload[
                "scheduler_state_dict"
            ]
        )

        scaler_state = resume_payload.get(
            "scaler_state_dict"
        )

        if scaler_state is not None:
            scaler.load_state_dict(
                scaler_state
            )

        best_epoch = int(
            resume_payload[
                "best_epoch"
            ]
        )

        best_patient_metrics = (
            resume_payload[
                "best_patient_metrics"
            ]
        )

        best_image_metrics = (
            resume_payload[
                "best_image_metrics"
            ]
        )

        patience_counter = int(
            resume_payload[
                "patience_counter"
            ]
        )

        history = resume_payload.get(
            "history",
            [],
        )

        completed_epoch = int(
            resume_payload[
                "completed_epoch"
            ]
        )

        start_epoch = (
            completed_epoch + 1
        )

        training_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "training_generator_state"
                ],

                state_name=(
                    "Training DataLoader generator state"
                ),
            )
        )

        validation_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "validation_generator_state"
                ],

                state_name=(
                    "Validation DataLoader generator state"
                ),
            )
        )

        restore_rng_state(
            resume_payload[
                "rng_state"
            ]
        )

        pretrained_cfg = resume_payload.get(
            "pretrained_cfg",
            pretrained_cfg,
        )

        repair_best_checkpoint = False

        if not best_checkpoint_path.exists():
            repair_best_checkpoint = True

        else:
            try:
                existing_best = safe_torch_load(
                    best_checkpoint_path,
                    map_location="cpu",
                )

                if (
                    existing_best.get(
                        "fold_fingerprint"
                    )
                    != fold_fingerprint
                ):
                    repair_best_checkpoint = True

                elif (
                    int(
                        existing_best.get(
                            "best_epoch",
                            -1,
                        )
                    )
                    != best_epoch
                ):
                    repair_best_checkpoint = True

                del existing_best

            except Exception:
                repair_best_checkpoint = True

        if repair_best_checkpoint:
            if best_epoch == completed_epoch:
                repaired_best_payload = {
                    "protocol_version": (
                        PROTOCOL_VERSION
                    ),

                    "analysis_plan_fingerprint": (
                        analysis_plan_fingerprint
                    ),

                    "fold_fingerprint": (
                        fold_fingerprint
                    ),

                    "experiment_name": (
                        experiment_name
                    ),

                    "training_seed": int(
                        training_seed
                    ),

                    "fold": int(fold),

                    "best_epoch": int(
                        best_epoch
                    ),

                    "model_state_dict": {
                        key: value.detach().cpu()
                        for key, value in (
                            resume_payload[
                                "model_state_dict"
                            ].items()
                        )
                    },

                    "best_patient_metrics": (
                        best_patient_metrics
                    ),

                    "best_image_metrics": (
                        best_image_metrics
                    ),

                    "pretrained_cfg": (
                        pretrained_cfg
                    ),
                }

                atomic_torch_save(
                    repaired_best_payload,
                    best_checkpoint_path,
                )

                print(
                    "Repaired best checkpoint from "
                    "the committed resume state."
                )

            else:
                raise RuntimeError(
                    "The best checkpoint is missing or "
                    "inconsistent and cannot be reconstructed.\n"
                    f"Experiment: {experiment_name}\n"
                    f"Seed: {training_seed}\n"
                    f"Fold: {fold}\n"
                    f"Best epoch: {best_epoch}\n"
                    f"Completed epoch: {completed_epoch}"
                )

        print(
            "Resuming from completed epoch:",
            completed_epoch,
        )

        print(
            "Next epoch:",
            start_epoch,
        )

        print(
            "Current patience:",
            patience_counter,
            "/",
            EARLY_STOP_PATIENCE,
        )

    training_required = (
        start_epoch <= EPOCHS
        and patience_counter
        < EARLY_STOP_PATIENCE
    )

    if training_required:
        for epoch in range(
            start_epoch,
            EPOCHS + 1,
        ):
            training_metrics = train_one_epoch(
                model=model,
                loader=training_loader,
                criterion=criterion,
                optimizer=optimizer,
                scaler=scaler,
                device=device,
            )

            (
                validation_image_metrics,
                validation_patient_metrics,
                _,
                _,
            ) = evaluate(
                model=model,
                loader=validation_loader,
                criterion=criterion,
                device=device,
            )

            current_lr = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )

            history_row = {
                "epoch": int(epoch),

                "lr": current_lr,

                "training_loss": float(
                    training_metrics[
                        "loss"
                    ]
                ),

                "training_image_qwk": float(
                    training_metrics[
                        "qwk"
                    ]
                ),

                "training_image_mae": float(
                    training_metrics[
                        "mae"
                    ]
                ),

                "training_image_macro_f1": float(
                    training_metrics[
                        "macro_f1"
                    ]
                ),

                "training_image_accuracy": float(
                    training_metrics[
                        "accuracy"
                    ]
                ),

                "validation_loss": float(
                    validation_image_metrics[
                        "loss"
                    ]
                ),

                "validation_image_qwk": float(
                    validation_image_metrics[
                        "qwk"
                    ]
                ),

                "validation_image_mae": float(
                    validation_image_metrics[
                        "mae"
                    ]
                ),

                "validation_image_macro_f1": float(
                    validation_image_metrics[
                        "macro_f1"
                    ]
                ),

                "validation_image_accuracy": float(
                    validation_image_metrics[
                        "accuracy"
                    ]
                ),

                "validation_patient_qwk": float(
                    validation_patient_metrics[
                        "qwk"
                    ]
                ),

                "validation_patient_mae": float(
                    validation_patient_metrics[
                        "mae"
                    ]
                ),

                "validation_patient_macro_f1": float(
                    validation_patient_metrics[
                        "macro_f1"
                    ]
                ),

                "validation_patient_accuracy": float(
                    validation_patient_metrics[
                        "accuracy"
                    ]
                ),
            }

            history.append(
                history_row
            )

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"lr={current_lr:.6f} | "
                f"patient QWK="
                f"{validation_patient_metrics['qwk']:.4f} | "
                f"patient MAE="
                f"{validation_patient_metrics['mae']:.4f} | "
                f"patient F1="
                f"{validation_patient_metrics['macro_f1']:.4f} | "
                f"image QWK="
                f"{validation_image_metrics['qwk']:.4f}"
            )

            improved = is_better_checkpoint(
                candidate_patient_metrics=(
                    validation_patient_metrics
                ),

                best_patient_metrics=(
                    best_patient_metrics
                ),

                candidate_image_metrics=(
                    validation_image_metrics
                ),

                best_image_metrics=(
                    best_image_metrics
                ),
            )

            if improved:
                best_epoch = int(epoch)

                best_patient_metrics = (
                    copy.deepcopy(
                        validation_patient_metrics
                    )
                )

                best_image_metrics = (
                    copy.deepcopy(
                        validation_image_metrics
                    )
                )

                patience_counter = 0

                print(
                    "  New best validation checkpoint."
                )

            else:
                patience_counter += 1

                print(
                    "  No improvement. Patience "
                    f"{patience_counter}/"
                    f"{EARLY_STOP_PATIENCE}"
                )

            atomic_csv_save(
                pd.DataFrame(history),
                history_path,
                index=False,
            )

            scheduler.step()

            if (
                epoch
                %
                EPOCH_CHECKPOINT_INTERVAL
                == 0
            ):
                resume_checkpoint_payload = {
                    "protocol_version": (
                        PROTOCOL_VERSION
                    ),

                    "analysis_plan_fingerprint": (
                        analysis_plan_fingerprint
                    ),

                    "fold_fingerprint": (
                        fold_fingerprint
                    ),

                    "experiment_name": (
                        experiment_name
                    ),

                    "experiment_config": (
                        experiment_config
                    ),

                    "training_seed": int(
                        training_seed
                    ),

                    "fold": int(fold),

                    "fold_seed": int(
                        fold_seed
                    ),

                    "completed_epoch": int(
                        epoch
                    ),

                    "best_epoch": int(
                        best_epoch
                    ),

                    "best_patient_metrics": (
                        best_patient_metrics
                    ),

                    "best_image_metrics": (
                        best_image_metrics
                    ),

                    "patience_counter": int(
                        patience_counter
                    ),

                    "history": history,

                    "model_state_dict": (
                        model.state_dict()
                    ),

                    "optimizer_state_dict": (
                        optimizer.state_dict()
                    ),

                    "scheduler_state_dict": (
                        scheduler.state_dict()
                    ),

                    "scaler_state_dict": (
                        scaler.state_dict()
                    ),

                    "rng_state": (
                        capture_rng_state()
                    ),

                    "training_generator_state": (
                        training_generator
                        .get_state()
                        .cpu()
                        .contiguous()
                    ),

                    "validation_generator_state": (
                        validation_generator
                        .get_state()
                        .cpu()
                        .contiguous()
                    ),

                    "pretrained_cfg": (
                        pretrained_cfg
                    ),

                    "current_epoch_is_best": bool(
                        improved
                    ),
                }

                atomic_torch_save(
                    resume_checkpoint_payload,
                    last_checkpoint_path,
                )

                print(
                    "  Saved epoch-resume checkpoint."
                )

            if improved:
                best_checkpoint_payload = {
                    "protocol_version": (
                        PROTOCOL_VERSION
                    ),

                    "analysis_plan_fingerprint": (
                        analysis_plan_fingerprint
                    ),

                    "fold_fingerprint": (
                        fold_fingerprint
                    ),

                    "experiment_name": (
                        experiment_name
                    ),

                    "training_seed": int(
                        training_seed
                    ),

                    "fold": int(fold),

                    "best_epoch": int(epoch),

                    "model_state_dict": {
                        key: value.detach().cpu()
                        for key, value in (
                            model.state_dict().items()
                        )
                    },

                    "best_patient_metrics": (
                        best_patient_metrics
                    ),

                    "best_image_metrics": (
                        best_image_metrics
                    ),

                    "pretrained_cfg": (
                        pretrained_cfg
                    ),
                }

                atomic_torch_save(
                    best_checkpoint_payload,
                    best_checkpoint_path,
                )

                print(
                    "  Saved best-model checkpoint."
                )

            if (
                patience_counter
                >= EARLY_STOP_PATIENCE
            ):
                print(
                    "Early stopping triggered."
                )

                break

    else:
        print(
            "Training already reached its stopping condition. "
            "Proceeding to best-checkpoint evaluation."
        )

    if not best_checkpoint_path.exists():
        raise RuntimeError(
            "No best checkpoint exists for:\n"
            f"{experiment_name}, "
            f"seed={training_seed}, "
            f"fold={fold}"
        )

    best_checkpoint = safe_torch_load(
        best_checkpoint_path,
        map_location="cpu",
    )

    if (
        best_checkpoint.get(
            "fold_fingerprint"
        )
        != fold_fingerprint
    ):
        raise RuntimeError(
            "Best checkpoint fingerprint mismatch."
        )

    model.load_state_dict(
        best_checkpoint[
            "model_state_dict"
        ]
    )

    (
        final_image_metrics,
        final_patient_metrics,
        final_image_prediction_df,
        final_patient_prediction_df,
    ) = evaluate(
        model=model,
        loader=validation_loader,
        criterion=criterion,
        device=device,
    )

    final_image_prediction_df[
        "fold"
    ] = int(fold)

    final_image_prediction_df[
        "training_seed"
    ] = int(training_seed)

    final_image_prediction_df = (
        validate_image_predictions(
            prediction_df=(
                final_image_prediction_df
            ),

            expected_df=validation_df,

            expected_seed=training_seed,

            expected_fold=fold,
        )
    )

    final_patient_prediction_df[
        "fold"
    ] = int(fold)

    final_patient_prediction_df[
        "training_seed"
    ] = int(training_seed)

    atomic_csv_save(
        final_image_prediction_df,
        image_prediction_path,
        index=False,
    )

    atomic_csv_save(
        final_patient_prediction_df,
        patient_prediction_path,
        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            final_image_prediction_df[
                "y_true"
            ],

            final_image_prediction_df[
                "y_pred"
            ],
        ),

        fold_directory
        / "validation_image_confusion_matrix.csv",

        index=True,
    )

    atomic_csv_save(
        make_confusion_df(
            final_patient_prediction_df[
                "y_true"
            ],

            final_patient_prediction_df[
                "y_pred"
            ],
        ),

        fold_directory
        / "validation_patient_confusion_matrix.csv",

        index=True,
    )

    fold_result = {
        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "protocol_version": PROTOCOL_VERSION,

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "fold_fingerprint": (
            fold_fingerprint
        ),

        "experiment_name": (
            experiment_name
        ),

        "experiment_display_name": (
            experiment_config[
                "display_name"
            ]
        ),

        "training_seed": int(
            training_seed
        ),

        "fold": int(fold),

        "fold_seed": int(
            fold_seed
        ),

        "best_epoch": int(
            best_checkpoint[
                "best_epoch"
            ]
        ),

        "parameter_count": int(
            parameter_count
        ),

        "image_metrics": {
            key: float(value)
            for key, value in (
                final_image_metrics.items()
            )
        },

        "patient_metrics": {
            key: float(value)
            for key, value in (
                final_patient_metrics.items()
            )
        },

        "validation_patient_class_counts": {
            str(class_index): int(
                validation_patient_distribution.loc[
                    class_index
                ]
            )
            for class_index in range(
                NUM_CLASSES
            )
        },

        "training_image_class_counts": (
            training_class_counts
        ),

        "training_image_class_weights": (
            training_class_weights
        ),

        "best_checkpoint_path": str(
            best_checkpoint_path
        ),

        "history_path": str(
            history_path
        ),

        "image_prediction_path": str(
            image_prediction_path
        ),

        "patient_prediction_path": str(
            patient_prediction_path
        ),

        "runtime_num_workers": int(
            RUNTIME_NUM_WORKERS
        ),

        "local_image_cache_enabled": bool(
            USE_LOCAL_IMAGE_CACHE
        ),

        "official_test_loaded": False,
        "official_test_evaluated": False,
    }

    atomic_json_save(
        fold_result,
        fold_result_path,
    )

    if last_checkpoint_path.exists():
        last_checkpoint_path.unlink()

    print(
        "Fold completed and finalized:",
        experiment_name,
        "| seed",
        training_seed,
        "| fold",
        fold,
    )

    del model
    del criterion
    del optimizer
    del scheduler
    del scaler
    del best_checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "fold_result": fold_result,

        "image_predictions": (
            final_image_prediction_df
        ),

        "patient_predictions": (
            final_patient_prediction_df
        ),
    }


# ============================================================
# 29. Stratified patient bootstrap
# ============================================================

def stratified_patient_bootstrap(
    patient_df: pd.DataFrame,
    repetitions: int,
    seed: int,
):
    rng = np.random.default_rng(
        seed
    )

    class_frames = {
        class_index: (
            patient_df[
                patient_df["y_true"]
                == class_index
            ]
            .reset_index(drop=True)
        )
        for class_index in range(
            NUM_CLASSES
        )
    }

    for class_index, class_df in (
        class_frames.items()
    ):
        if len(class_df) == 0:
            raise RuntimeError(
                "Bootstrap cannot proceed because "
                f"class {class_index} is empty."
            )

    bootstrap_rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_parts = []

        for class_index in range(
            NUM_CLASSES
        ):
            class_df = class_frames[
                class_index
            ]

            sampled_indices = rng.integers(
                low=0,
                high=len(class_df),
                size=len(class_df),
            )

            sampled_parts.append(
                class_df.iloc[
                    sampled_indices
                ]
            )

        sampled_df = pd.concat(
            sampled_parts,
            ignore_index=True,
        )

        metrics = compute_metrics(
            sampled_df["y_true"],
            sampled_df["y_pred"],
        )

        bootstrap_rows.append({
            "bootstrap_index": int(
                bootstrap_index
            ),

            **metrics,
        })

    bootstrap_df = pd.DataFrame(
        bootstrap_rows
    )

    interval_rows = []

    for metric_name in [
        "qwk",
        "mae",
        "macro_f1",
        "accuracy",
    ]:
        values = (
            bootstrap_df[
                metric_name
            ]
            .dropna()
            .to_numpy()
        )

        interval_rows.append({
            "metric": metric_name,

            "lower_95": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "upper_95": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

            "n_valid_bootstrap": int(
                len(values)
            ),
        })

    return (
        pd.DataFrame(
            interval_rows
        ),

        bootstrap_df,
    )


# ============================================================
# 30. Paired patient bootstrap
# ============================================================

def paired_patient_bootstrap(
    patient_df_a: pd.DataFrame,
    patient_df_b: pd.DataFrame,
    repetitions: int,
    seed: int,
    candidate_a: str,
    candidate_b: str,
):
    aligned_df = (
        patient_df_a[
            [
                "patient_id",
                "y_true",
                "y_pred",
            ]
        ]
        .rename(
            columns={
                "y_pred": "y_pred_a"
            }
        )
        .merge(
            patient_df_b[
                [
                    "patient_id",
                    "y_true",
                    "y_pred",
                ]
            ]
            .rename(
                columns={
                    "y_true": "y_true_b",
                    "y_pred": "y_pred_b",
                }
            ),
            on="patient_id",
            how="inner",
            validate="one_to_one",
        )
    )

    if len(aligned_df) != 478:
        raise RuntimeError(
            "Paired comparison did not align "
            "all 478 patients."
        )

    if not (
        aligned_df["y_true"]
        ==
        aligned_df["y_true_b"]
    ).all():
        raise RuntimeError(
            "Patient references differ between "
            "the paired prediction sets."
        )

    aligned_df = aligned_df.drop(
        columns=[
            "y_true_b"
        ]
    )

    point_metrics_a = compute_metrics(
        aligned_df["y_true"],
        aligned_df["y_pred_a"],
    )

    point_metrics_b = compute_metrics(
        aligned_df["y_true"],
        aligned_df["y_pred_b"],
    )

    class_frames = {
        class_index: (
            aligned_df[
                aligned_df["y_true"]
                == class_index
            ]
            .reset_index(drop=True)
        )
        for class_index in range(
            NUM_CLASSES
        )
    }

    rng = np.random.default_rng(
        seed
    )

    bootstrap_rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_parts = []

        for class_index in range(
            NUM_CLASSES
        ):
            class_df = class_frames[
                class_index
            ]

            sampled_indices = rng.integers(
                low=0,
                high=len(class_df),
                size=len(class_df),
            )

            sampled_parts.append(
                class_df.iloc[
                    sampled_indices
                ]
            )

        sampled_df = pd.concat(
            sampled_parts,
            ignore_index=True,
        )

        metrics_a = compute_metrics(
            sampled_df["y_true"],
            sampled_df["y_pred_a"],
        )

        metrics_b = compute_metrics(
            sampled_df["y_true"],
            sampled_df["y_pred_b"],
        )

        bootstrap_rows.append({
            "bootstrap_index": int(
                bootstrap_index
            ),

            "delta_qwk": (
                metrics_a["qwk"]
                - metrics_b["qwk"]
            ),

            "delta_mae": (
                metrics_a["mae"]
                - metrics_b["mae"]
            ),

            "delta_macro_f1": (
                metrics_a["macro_f1"]
                - metrics_b["macro_f1"]
            ),

            "delta_accuracy": (
                metrics_a["accuracy"]
                - metrics_b["accuracy"]
            ),
        })

    bootstrap_df = pd.DataFrame(
        bootstrap_rows
    )

    metric_mapping = {
        "qwk": "delta_qwk",
        "mae": "delta_mae",
        "macro_f1": "delta_macro_f1",
        "accuracy": "delta_accuracy",
    }

    summary_rows = []

    for metric_name, difference_column in (
        metric_mapping.items()
    ):
        values = (
            bootstrap_df[
                difference_column
            ]
            .dropna()
            .to_numpy()
        )

        point_difference = (
            point_metrics_a[
                metric_name
            ]
            -
            point_metrics_b[
                metric_name
            ]
        )

        probability_nonpositive = float(
            np.mean(values <= 0)
        )

        probability_nonnegative = float(
            np.mean(values >= 0)
        )

        two_sided_tail_proportion = min(
            1.0,
            2.0
            *
            min(
                probability_nonpositive,
                probability_nonnegative,
            ),
        )

        summary_rows.append({
            "candidate_a": candidate_a,
            "candidate_b": candidate_b,

            "difference_definition": (
                f"{candidate_a} - {candidate_b}"
            ),

            "metric": metric_name,

            "point_difference": float(
                point_difference
            ),

            "lower_95": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "upper_95": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

            "bootstrap_two_sided_tail_proportion": float(
                two_sided_tail_proportion
            ),

            "direction_note": (
                "Positive favors candidate A"
                if metric_name in [
                    "qwk",
                    "macro_f1",
                    "accuracy",
                ]
                else
                "Negative favors candidate A"
            ),
        })

    return (
        pd.DataFrame(
            summary_rows
        ),

        bootstrap_df,
    )


# ============================================================
# 31. Run one complete loss experiment
# ============================================================

def run_experiment(
    experiment_name: str,
    preset: dict,
):
    experiment_config = copy.deepcopy(
        BASE_TRAINING_CONFIG
    )

    experiment_config.update(
        preset
    )

    experiment_config[
        "experiment_name"
    ] = experiment_name

    experiment_config[
        "analysis_plan_fingerprint"
    ] = analysis_plan_fingerprint

    experiment_fingerprint = (
        dictionary_sha256(
            experiment_config
        )
    )

    experiment_directory = (
        OUTPUT_ROOT
        / experiment_name
        / f"seed_{SCREENING_SEED}"
    )

    experiment_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    config_path = (
        experiment_directory
        / "config_used.json"
    )

    result_row_path = (
        experiment_directory
        / "result_row.json"
    )

    summary_path = (
        experiment_directory
        / "summary.json"
    )

    oof_image_path = (
        experiment_directory
        / "oof_image_predictions.csv"
    )

    oof_patient_path = (
        experiment_directory
        / "oof_patient_predictions.csv"
    )

    bootstrap_ci_path = (
        experiment_directory
        / "oof_patient_bootstrap_ci.csv"
    )

    bootstrap_samples_path = (
        experiment_directory
        / "oof_patient_bootstrap_samples.csv"
    )

    completed_experiment_paths = [
        result_row_path,
        summary_path,
        oof_image_path,
        oof_patient_path,
        bootstrap_ci_path,
        bootstrap_samples_path,
    ]

    if (
        SKIP_COMPLETED_EXPERIMENTS
        and all(
            path.exists()
            for path in (
                completed_experiment_paths
            )
        )
    ):
        with open(
            result_row_path,
            "r",
            encoding="utf-8",
        ) as file:
            completed_result = json.load(
                file
            )

        if (
            completed_result.get(
                "experiment_fingerprint"
            )
            != experiment_fingerprint
        ):
            raise RuntimeError(
                "Completed experiment fingerprint mismatch:\n"
                f"{experiment_directory}"
            )

        completed_image_df = pd.read_csv(
            oof_image_path
        )

        completed_image_df = (
            validate_image_predictions(
                prediction_df=(
                    completed_image_df
                ),

                expected_df=trainval_df,

                expected_seed=SCREENING_SEED,

                expected_fold=None,
            )
        )

        completed_patient_df = (
            aggregate_mean_probabilities(
                completed_image_df
            )
        )

        print(
            "\nLoaded completed experiment:",
            experiment_name,
        )

        return {
            "result_row": completed_result,

            "oof_image_df": (
                completed_image_df
            ),

            "oof_patient_df": (
                completed_patient_df
            ),
        }

    atomic_json_save(
        experiment_config,
        config_path,
    )

    fold_outputs = []
    fold_prediction_parts = []
    fold_summary_rows = []

    for fold in range(
        NUM_FOLDS
    ):
        fold_output = (
            run_or_resume_fold(
                experiment_name=(
                    experiment_name
                ),

                experiment_config=(
                    experiment_config
                ),

                training_seed=(
                    SCREENING_SEED
                ),

                fold=fold,

                experiment_directory=(
                    experiment_directory
                ),
            )
        )

        fold_outputs.append(
            fold_output
        )

        fold_prediction_parts.append(
            fold_output[
                "image_predictions"
            ]
        )

        fold_result = (
            fold_output[
                "fold_result"
            ]
        )

        fold_summary_row = {
            "experiment_name": (
                experiment_name
            ),

            "training_seed": int(
                SCREENING_SEED
            ),

            "fold": int(fold),

            "best_epoch": int(
                fold_result[
                    "best_epoch"
                ]
            ),

            "validation_image_qwk": float(
                fold_result[
                    "image_metrics"
                ]["qwk"]
            ),

            "validation_image_mae": float(
                fold_result[
                    "image_metrics"
                ]["mae"]
            ),

            "validation_image_macro_f1": float(
                fold_result[
                    "image_metrics"
                ]["macro_f1"]
            ),

            "validation_image_accuracy": float(
                fold_result[
                    "image_metrics"
                ]["accuracy"]
            ),

            "validation_patient_qwk": float(
                fold_result[
                    "patient_metrics"
                ]["qwk"]
            ),

            "validation_patient_mae": float(
                fold_result[
                    "patient_metrics"
                ]["mae"]
            ),

            "validation_patient_macro_f1": float(
                fold_result[
                    "patient_metrics"
                ]["macro_f1"]
            ),

            "validation_patient_accuracy": float(
                fold_result[
                    "patient_metrics"
                ]["accuracy"]
            ),
        }

        for class_index in range(
            NUM_CLASSES
        ):
            fold_summary_row[
                f"validation_patient_count_mayo_{class_index}"
            ] = int(
                fold_result[
                    "validation_patient_class_counts"
                ][str(class_index)]
            )

        fold_summary_rows.append(
            fold_summary_row
        )

    oof_image_df = (
        pd.concat(
            fold_prediction_parts,
            ignore_index=True,
        )
        .sort_values("path")
        .reset_index(drop=True)
    )

    oof_image_df = (
        validate_image_predictions(
            prediction_df=oof_image_df,

            expected_df=trainval_df,

            expected_seed=SCREENING_SEED,

            expected_fold=None,
        )
    )

    if "fold" not in oof_image_df.columns:
        raise RuntimeError(
            "OOF predictions do not contain "
            "fold identifiers."
        )

    if (
        oof_image_df
        .groupby("patient_id")["fold"]
        .nunique()
        .max()
        != 1
    ):
        raise RuntimeError(
            "A patient appears in multiple OOF folds."
        )

    oof_patient_df = (
        aggregate_mean_probabilities(
            oof_image_df
        )
    )

    if len(oof_patient_df) != 478:
        raise RuntimeError(
            "OOF patient prediction count is not 478."
        )

    audit_columns = [
        "patient_id",
        "modal_low",
        "modal_high",
        "median_low",
        "median_high",
        "maximum_label",
        "n_images",
        "n_unique_labels",
        "label_range",
        "modal_tie",
        "majority_fraction",
        "label_entropy",
    ]

    available_audit_columns = [
        column
        for column in audit_columns
        if column in (
            reference_audit_df.columns
        )
    ]

    oof_patient_df = (
        oof_patient_df
        .merge(
            reference_audit_df[
                available_audit_columns
            ],

            on="patient_id",

            how="left",

            validate="one_to_one",

            suffixes=(
                "",
                "_audit",
            ),
        )
    )

    if (
        "n_images_audit"
        in oof_patient_df.columns
    ):
        if not (
            oof_patient_df["n_images"]
            ==
            oof_patient_df[
                "n_images_audit"
            ]
        ).all():
            raise RuntimeError(
                "OOF patient image counts do not "
                "match the reference audit."
            )

        oof_patient_df = (
            oof_patient_df.drop(
                columns=[
                    "n_images_audit"
                ]
            )
        )

    if not (
        oof_patient_df[
            "y_true"
        ].astype(int)
        ==
        oof_patient_df[
            PRIMARY_REFERENCE_RULE
        ].astype(int)
    ).all():
        raise RuntimeError(
            "OOF patient labels do not match modal_low."
        )

    oof_image_metrics = compute_metrics(
        oof_image_df["y_true"],
        oof_image_df["y_pred"],
    )

    oof_patient_metrics = compute_metrics(
        oof_patient_df["y_true"],
        oof_patient_df["y_pred"],
    )

    (
        bootstrap_ci_df,
        bootstrap_samples_df,
    ) = stratified_patient_bootstrap(
        patient_df=oof_patient_df,

        repetitions=BOOTSTRAP_REPETITIONS,

        seed=(
            BOOTSTRAP_SEED
            +
            list(
                EXPERIMENT_PRESETS.keys()
            ).index(
                experiment_name
            )
        ),
    )

    point_estimate_map = {
        "qwk": (
            oof_patient_metrics["qwk"]
        ),

        "mae": (
            oof_patient_metrics["mae"]
        ),

        "macro_f1": (
            oof_patient_metrics[
                "macro_f1"
            ]
        ),

        "accuracy": (
            oof_patient_metrics[
                "accuracy"
            ]
        ),
    }

    bootstrap_ci_df.insert(
        1,
        "point_estimate",
        bootstrap_ci_df[
            "metric"
        ].map(
            point_estimate_map
        ),
    )

    atomic_csv_save(
        oof_image_df,
        oof_image_path,
        index=False,
    )

    atomic_csv_save(
        oof_patient_df,
        oof_patient_path,
        index=False,
    )

    atomic_csv_save(
        bootstrap_ci_df,
        bootstrap_ci_path,
        index=False,
    )

    atomic_csv_save(
        bootstrap_samples_df,
        bootstrap_samples_path,
        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            oof_image_df["y_true"],
            oof_image_df["y_pred"],
        ),

        experiment_directory
        / "oof_image_confusion_matrix.csv",

        index=True,
    )

    atomic_csv_save(
        make_confusion_df(
            oof_patient_df["y_true"],
            oof_patient_df["y_pred"],
        ),

        experiment_directory
        / "oof_patient_confusion_matrix.csv",

        index=True,
    )

    fold_summary_df = pd.DataFrame(
        fold_summary_rows
    )

    fold_summary_path = (
        experiment_directory
        / "fold_summary.csv"
    )

    atomic_csv_save(
        fold_summary_df,
        fold_summary_path,
        index=False,
    )

    first_best_checkpoint_path = Path(
        fold_outputs[0][
            "fold_result"
        ][
            "best_checkpoint_path"
        ]
    )

    first_best_checkpoint = safe_torch_load(
        first_best_checkpoint_path,
        map_location="cpu",
    )

    pretrained_cfg_path = (
        experiment_directory
        / "pretrained_cfg.json"
    )

    atomic_json_save(
        first_best_checkpoint.get(
            "pretrained_cfg",
            {},
        ),

        pretrained_cfg_path,
    )

    del first_best_checkpoint

    ci_lookup = (
        bootstrap_ci_df
        .set_index("metric")
    )

    result_row = {
        "experiment_name": (
            experiment_name
        ),

        "display_name": (
            preset["display_name"]
        ),

        "training_seed": int(
            SCREENING_SEED
        ),

        "experiment_fingerprint": (
            experiment_fingerprint
        ),

        "model_name": (
            RESOLVED_SWIN_MODEL_NAME
        ),

        "loss_type": (
            experiment_config[
                "loss_type"
            ]
        ),

        "ordinal_loss_weight": float(
            experiment_config[
                "ordinal_loss_weight"
            ]
        ),

        "ordinal_alpha": float(
            experiment_config[
                "ordinal_alpha"
            ]
        ),

        "patient_reference": (
            PRIMARY_REFERENCE_RULE
        ),

        "patient_aggregation": (
            LOCKED_AGGREGATION_RULE
        ),

        "oof_image_qwk": float(
            oof_image_metrics["qwk"]
        ),

        "oof_image_mae": float(
            oof_image_metrics["mae"]
        ),

        "oof_image_macro_f1": float(
            oof_image_metrics[
                "macro_f1"
            ]
        ),

        "oof_image_accuracy": float(
            oof_image_metrics[
                "accuracy"
            ]
        ),

        "oof_patient_qwk": float(
            oof_patient_metrics["qwk"]
        ),

        "oof_patient_qwk_ci_low": float(
            ci_lookup.loc[
                "qwk",
                "lower_95",
            ]
        ),

        "oof_patient_qwk_ci_high": float(
            ci_lookup.loc[
                "qwk",
                "upper_95",
            ]
        ),

        "oof_patient_mae": float(
            oof_patient_metrics["mae"]
        ),

        "oof_patient_mae_ci_low": float(
            ci_lookup.loc[
                "mae",
                "lower_95",
            ]
        ),

        "oof_patient_mae_ci_high": float(
            ci_lookup.loc[
                "mae",
                "upper_95",
            ]
        ),

        "oof_patient_macro_f1": float(
            oof_patient_metrics[
                "macro_f1"
            ]
        ),

        "oof_patient_macro_f1_ci_low": float(
            ci_lookup.loc[
                "macro_f1",
                "lower_95",
            ]
        ),

        "oof_patient_macro_f1_ci_high": float(
            ci_lookup.loc[
                "macro_f1",
                "upper_95",
            ]
        ),

        "oof_patient_accuracy": float(
            oof_patient_metrics[
                "accuracy"
            ]
        ),

        "oof_patient_accuracy_ci_low": float(
            ci_lookup.loc[
                "accuracy",
                "lower_95",
            ]
        ),

        "oof_patient_accuracy_ci_high": float(
            ci_lookup.loc[
                "accuracy",
                "upper_95",
            ]
        ),

        "official_test_loaded": False,
        "official_test_evaluated": False,

        "experiment_directory": str(
            experiment_directory
        ),
    }

    summary = {
        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "experiment_name": (
            experiment_name
        ),

        "display_name": (
            preset["display_name"]
        ),

        "experiment_fingerprint": (
            experiment_fingerprint
        ),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "selection_data": (
            "Official LIMUC development partition only"
        ),

        "checkpoint_selection": (
            "Validation patient QWK, then patient MAE, "
            "patient macro F1, and image QWK"
        ),

        "patient_reference": (
            PRIMARY_REFERENCE_RULE
        ),

        "patient_aggregation": (
            LOCKED_AGGREGATION_RULE
        ),

        "oof_image_metrics": (
            oof_image_metrics
        ),

        "oof_patient_metrics": (
            oof_patient_metrics
        ),

        "oof_patient_95_ci": {
            metric_name: [
                float(
                    ci_lookup.loc[
                        metric_name,
                        "lower_95",
                    ]
                ),

                float(
                    ci_lookup.loc[
                        metric_name,
                        "upper_95",
                    ]
                ),
            ]
            for metric_name in [
                "qwk",
                "mae",
                "macro_f1",
                "accuracy",
            ]
        },

        "runtime_num_workers": int(
            RUNTIME_NUM_WORKERS
        ),

        "local_image_cache_enabled": bool(
            USE_LOCAL_IMAGE_CACHE
        ),

        "official_test_loaded": False,
        "official_test_evaluated": False,
    }

    atomic_json_save(
        result_row,
        result_row_path,
    )

    atomic_json_save(
        summary,
        summary_path,
    )

    print("\nOOF image metrics:")

    print(
        json.dumps(
            oof_image_metrics,
            indent=2,
        )
    )

    print("\nOOF patient metrics:")

    print(
        json.dumps(
            oof_patient_metrics,
            indent=2,
        )
    )

    print(
        "\nOOF patient bootstrap intervals:"
    )

    display(
        bootstrap_ci_df
    )

    return {
        "result_row": result_row,
        "oof_image_df": oof_image_df,
        "oof_patient_df": oof_patient_df,
    }


# ============================================================
# 32. Execute all five loss candidates
# ============================================================

experiment_outputs = {}
result_rows = []
patient_prediction_lookup = {}

for experiment_name in (
    EXPERIMENTS_TO_RUN
):
    experiment_output = run_experiment(
        experiment_name=experiment_name,

        preset=(
            EXPERIMENT_PRESETS[
                experiment_name
            ]
        ),
    )

    experiment_outputs[
        experiment_name
    ] = experiment_output

    result_rows.append(
        experiment_output[
            "result_row"
        ]
    )

    patient_prediction_lookup[
        experiment_name
    ] = experiment_output[
        "oof_patient_df"
    ]


# ============================================================
# 33. Rank Swin losses using development OOF metrics
# ============================================================

results_df = pd.DataFrame(
    result_rows
)

results_ranked_df = (
    results_df
    .sort_values(
        by=[
            "oof_patient_qwk",
            "oof_patient_mae",
            "oof_patient_macro_f1",
            "oof_image_qwk",
        ],

        ascending=[
            False,
            True,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

results_ranked_df.insert(
    0,
    "development_rank",
    np.arange(
        1,
        len(results_ranked_df) + 1,
    ),
)

results_path = (
    OUTPUT_ROOT
    / "swin_loss_screening_results.csv"
)

ranked_results_path = (
    OUTPUT_ROOT
    / "swin_loss_screening_ranked_development_only.csv"
)

atomic_csv_save(
    results_df,
    results_path,
    index=False,
)

atomic_csv_save(
    results_ranked_df,
    ranked_results_path,
    index=False,
)


# ============================================================
# 34. Create top-two shortlist
# ============================================================

shortlist_size = min(
    2,
    len(results_ranked_df),
)

shortlist_df = (
    results_ranked_df
    .head(shortlist_size)
    .copy()
)

shortlist_path = (
    OUTPUT_ROOT
    / "swin_loss_shortlist_for_multiseed_confirmation.csv"
)

atomic_csv_save(
    shortlist_df,
    shortlist_path,
    index=False,
)

TOP_CANDIDATE = str(
    shortlist_df.iloc[
        0
    ][
        "experiment_name"
    ]
)

SECOND_CANDIDATE = str(
    shortlist_df.iloc[
        1
    ][
        "experiment_name"
    ]
)


# ============================================================
# 35. Paired bootstrap between top two candidates
# ============================================================

(
    top2_paired_summary_df,
    top2_paired_samples_df,
) = paired_patient_bootstrap(
    patient_df_a=(
        patient_prediction_lookup[
            TOP_CANDIDATE
        ]
    ),

    patient_df_b=(
        patient_prediction_lookup[
            SECOND_CANDIDATE
        ]
    ),

    repetitions=BOOTSTRAP_REPETITIONS,

    seed=(
        BOOTSTRAP_SEED
        + 100000
    ),

    candidate_a=TOP_CANDIDATE,

    candidate_b=SECOND_CANDIDATE,
)

top2_paired_summary_path = (
    OUTPUT_ROOT
    / "swin_top2_paired_bootstrap_summary.csv"
)

top2_paired_samples_path = (
    OUTPUT_ROOT
    / "swin_top2_paired_bootstrap_samples.csv"
)

atomic_csv_save(
    top2_paired_summary_df,
    top2_paired_summary_path,
    index=False,
)

atomic_csv_save(
    top2_paired_samples_df,
    top2_paired_samples_path,
    index=False,
)


# ============================================================
# 36. Save compact screening decision
# ============================================================

screening_decision = {
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "protocol_version": PROTOCOL_VERSION,

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,

    "model_name": (
        RESOLVED_SWIN_MODEL_NAME
    ),

    "screening_seed": int(
        SCREENING_SEED
    ),

    "primary_patient_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "locked_patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),

    "top_ranked_screening_candidate": (
        TOP_CANDIDATE
    ),

    "second_ranked_screening_candidate": (
        SECOND_CANDIDATE
    ),

    "shortlisted_candidates": (
        shortlist_df[
            "experiment_name"
        ].tolist()
    ),

    "selection_hierarchy": [
        "Pooled OOF patient QWK: higher",
        "Pooled OOF patient MAE: lower",
        "Pooled OOF patient macro F1: higher",
        "Pooled OOF image QWK: higher",
    ],

    "interpretation": (
        "The top two loss configurations were selected "
        "using development-only pooled OOF metrics. "
        "The top-ranked single-seed candidate is not "
        "treated as final until multi-seed confirmation."
    ),

    "runtime_io_recovery": {
        "runtime_num_workers": int(
            RUNTIME_NUM_WORKERS
        ),

        "local_image_cache_enabled": bool(
            USE_LOCAL_IMAGE_CACHE
        ),

        "image_read_max_retries": int(
            IMAGE_IO_MAX_RETRIES
        ),
    },

    "output_files": {
        "ranked_results": str(
            ranked_results_path
        ),

        "shortlist": str(
            shortlist_path
        ),

        "top2_paired_bootstrap": str(
            top2_paired_summary_path
        ),
    },
}

screening_decision_path = (
    OUTPUT_ROOT
    / "swin_loss_screening_decision_summary.json"
)

atomic_json_save(
    screening_decision,
    screening_decision_path,
)


# ============================================================
# 37. Display results
# ============================================================

display_columns = [
    "development_rank",
    "experiment_name",
    "display_name",

    "oof_patient_qwk",
    "oof_patient_qwk_ci_low",
    "oof_patient_qwk_ci_high",

    "oof_patient_mae",
    "oof_patient_mae_ci_low",
    "oof_patient_mae_ci_high",

    "oof_patient_macro_f1",
    "oof_patient_accuracy",

    "oof_image_qwk",
    "oof_image_mae",
]

print(
    "\n"
    + "=" * 120
)

print(
    "DEVELOPMENT-ONLY SWIN LOSS RANKING"
)

print("=" * 120)

display(
    results_ranked_df[
        display_columns
    ]
)


print(
    "\n"
    + "=" * 120
)

print(
    "TOP-TWO SWIN LOSS SHORTLIST"
)

print("=" * 120)

display(
    shortlist_df[
        display_columns
    ]
)


print(
    "\n"
    + "=" * 120
)

print(
    "TOP-TWO PAIRED PATIENT BOOTSTRAP"
)

print("=" * 120)

display(
    top2_paired_summary_df
)


# ============================================================
# 38. Final status
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "04_swin_loss_screening_development_only COMPLETED"
)

print("=" * 120)

print(
    "Output directory:",
    OUTPUT_ROOT,
)

print(
    "Frozen analysis plan:",
    analysis_plan_path,
)

print(
    "Runtime I/O recovery record:",
    OUTPUT_ROOT
    / "runtime_google_drive_io_recovery.json",
)

print(
    "Ranked Swin results:",
    ranked_results_path,
)

print(
    "Top-two shortlist:",
    shortlist_path,
)

print(
    "Top-two paired bootstrap:",
    top2_paired_summary_path,
)

print(
    "Decision summary:",
    screening_decision_path,
)

print(
    "\nPrimary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Locked patient aggregation:",
    LOCKED_AGGREGATION_RULE,
)

print(
    "Official LIMUC test data loaded:",
    False,
)

print(
    "Official LIMUC test data evaluated:",
    False,
)

print(
    "\nRuntime DataLoader workers:",
    RUNTIME_NUM_WORKERS,
)

print(
    "Local image cache:",
    LOCAL_IMAGE_CACHE_ROOT,
)

print(
    "\nThe top-ranked single-seed Swin candidate "
    "is a screening result, not the final Swin decision."
)

print(
    "The top two candidates should next be evaluated "
    "using seeds 42, 2025, and 2026."
)